# Rwanda Forest Health Monitor V2 (nightly)
Runs daily at **02:10 Kigali time** on GeoHub.

Monitors Rwanda's **named forests** (Nyungwe, Akagera, Gishwati, Mukura, Volcanoes and others): the 2024 forest
cover of each area in the *Protected Areas* layer, without wetlands and forests under 50 ha.

Each run:
1. Finds new Sentinel-2 Level-2A dates (at most 20% cloud) in the last 14 months.
2. Runs four **Zonal Statistics** jobs (NDVI, NDMI, NBR, clear share) over the 20 m zone raster of the forests,
   on clear pixels only.
3. Writes `Observations` and updates each forest's condition, status and note in `ForestUnits`.
4. Refreshes `AdminLinks` (Province, District, Sector and Village filters).
5. Saves a new **overlay picture** when a forest has a newer clear image; the previous one is deleted.
6. Logs the run in `RunLog`.

**Runtime:** ArcGIS Notebook Python 3 Advanced with raster analytics.

In [ ]:
def _p(name, default):
    return globals().get(name, default)


RUN_MODE = _p("RUN_MODE", "nightly")            # nightly | dry_run (computes, writes nothing)

# ---- project ----
PROJECT_TITLE = "Rwanda Forest Health Monitor"
PROJECT_CODE = "fh"
SERVICE_NAME = _p("SERVICE_NAME", "RW_ForestHealth_V2")
FOLDER = _p("FOLDER", "Forest_health_monitoring_V2")
NOTEBOOK_TITLE = _p("NOTEBOOK_TITLE", "RW Forest Health Monitor V2 - Nightly")
WEBMAP_TITLE = "RW Forest Health V2 - Map"
DASHBOARD_TITLE = "Rwanda Forest Health Monitor V2"
DASHBOARD_SUBTITLE = "Canopy condition, disturbance and moisture stress of Rwanda's parks and natural forests from Sentinel-2, last 14 months"
TAGS = "Rwanda,forest,forest health,deforestation,national park,Sentinel-2,GeoHub,monitoring"
SERVICE_SNIPPET = ("Nightly Sentinel-2 forest health for Rwanda's named forests (national parks and natural forests): "
                   "condition, disturbance alerts, moisture stress and 14-month history.")
SERVICE_DESCRIPTION = ("Maintained automatically by the notebook '" + NOTEBOOK_TITLE + "'. ForestUnits holds one row per "
                       "named forest (the 2024 forest cover of each protected area of at least 50 ha) with the latest status; "
                       "Observations holds one row per forest per Sentinel-2 date; AdminLinks lists the villages each forest "
                       "covers, for dashboard filtering. Source of the forests: the Protected Areas layer (RDB, RFA, MoE).")

# ---- inputs (GeoHub item IDs / URLs) ----
S2_ITEM_ID = _p("S2_ITEM_ID", "db0c0c50190744bd8e4276ed752aa27d")        # Sentinel2L2A proxy (stored credentials)
ADMIN_ITEM_ID = _p("ADMIN_ITEM_ID", "b9ec668553aa4d8b8ef35b76a97c6fdf")  # Admin_Boundaries (2022)
SOURCE_URL = _p("SOURCE_URL", "https://gh.space.gov.rw/server/rest/services/Protected_Areas/MapServer/0")  # 2024 land cover per area
SOURCE_WHERE = _p("SOURCE_WHERE", "class = 'Forest' AND type <> 'Wetland' AND area_ha >= 50")
RESYNC_UNITS = _p("RESYNC_UNITS", False)       # True = reload the forests and admin links from the source layer
DRY_FORESTS = ["Akagera National Park", "Gabiro", "Karama natural forest", "Umuvumba NF", "Karangazi NF",
               "Ibanda_Makera", "Kibirizi Muyira 1", "Kibirizi Muyira 4"]   # source names; all others are montane forest

# ---- imagery and quality rules ----
SCENE_MAX_CLOUD = _p("SCENE_MAX_CLOUD", 0.20)   # tile-level cloud cover filter (0-1)
UNIT_MAX_CLOUD = _p("UNIT_MAX_CLOUD", 0.20)     # max cloud/shadow share over a forest
LARGE_FOREST_HA = 5000                           # forests at least this big ...
LARGE_MIN_CLEAR = 0.5                            # ... count an observation when half of them is clear
HISTORY_MONTHS = _p("HISTORY_MONTHS", 14)       # history window and first-run back-fill
MAX_DATES_PER_RUN = _p("MAX_DATES_PER_RUN", 40)  # the time budget stops earlier if needed
MAX_SCENE_ATTEMPTS = 3
ZONE_CELL_M = 20                                 # the zone raster: one cell per 20 m, value = the forest's number
STATS_SIMPLIFY_M = 100                           # outline detail used for the overlay picture
STATS_RING_SHARE = 0.99                          # the overlay is cut with the largest patches holding this share
STATS_MAX_RINGS = 400                            # ... and never more rings than this
MIN_COVERAGE = 0.6                               # share of the forest inside the day's tiles
WORKERS = _p("WORKERS", 8)
MAX_UNITS = _p("MAX_UNITS", 0)                   # >0 limits the number of forests (testing)
STALE_DAYS = 45

# ---- interpretation thresholds (provisional) ----
CONDITION_NDVI = {"Natural forest": (0.70, 0.55), "Dry forest & savannah": (0.40, 0.22)}   # (Good from, Fair from)
DISTURBANCE_NBR_DROP = 0.10
DISTURBANCE_NDVI_DROP = 0.08
MOISTURE_ANOMALY = -0.10
WATCH_NDVI_ANOMALY = -0.08

# ---- overlay picture: each forest on its latest clear image, saved as one hosted imagery layer ----
MAKE_OVERLAY = _p("MAKE_OVERLAY", True)          # False = draw the overlays live from the Sentinel-2 service
OVERLAY_FORCE = _p("OVERLAY_FORCE", False)       # True = make a new picture even if no forest has a newer clear image
OVERLAY_CELL_M = 20
OVERLAY_CHUNK_PX = 4000                          # pixels per side per request (the image service allows 5000)
OVERLAY_WORKERS = 4
OVERLAY_NODATA = -9999
OVERLAY_RESERVE_MIN = 45                         # postpone the picture when less time than this is left
OVERLAY_MIN_SCALE = 600000                       # the picture only draws when zoomed in past 1:600,000

# ---- apps and schedule ----
SHARE_TO = _p("SHARE_TO", "org")                # "org", "everyone", a group title, or "" to skip
ENSURE_APPS = _p("ENSURE_APPS", True)
REBUILD_APPS = _p("REBUILD_APPS", False)
KEEP_INTERMEDIATE = False
CREATE_TASK = _p("CREATE_TASK", False)          # True once, after saving this notebook as NOTEBOOK_TITLE
TASK_CRON_UTC = "10 0"                  # "minute hour" in UTC (Kigali = UTC+2)
TASK_LABEL = "daily 02:10 Kigali"
TASK_TIMEOUT_MIN = 720

## 1. Connect

In [ ]:
import os, re, json, time, math, uuid, statistics, traceback
import datetime as dt
from concurrent.futures import ThreadPoolExecutor, as_completed

from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection

UTC = dt.timezone.utc
KIGALI = dt.timezone(dt.timedelta(hours=2), "Africa/Kigali")


def connect():
    try:
        g = GIS("home")
        if g.users.me is None:
            raise RuntimeError("anonymous session")
        return g
    except Exception:
        return GIS(os.environ["ARCGIS_PORTAL_URL"], os.environ["ARCGIS_USERNAME"], os.environ["ARCGIS_PASSWORD"])


gis = connect()
ME = gis.users.me.username
RUN_ID = uuid.uuid4().hex[:12]
RUN_START = dt.datetime.now(UTC)
TODAY = RUN_START.astimezone(KIGALI).date()
DRY_RUN = RUN_MODE == "dry_run"
TIME_BUDGET_MIN = globals().get("TIME_BUDGET_MIN") or int(0.75 * globals().get("TASK_TIMEOUT_MIN", 300))
_LOG = []


def log(msg):
    line = f"{dt.datetime.now(KIGALI):%H:%M:%S} {msg}"
    print(line, flush=True)
    _LOG.append(line)


def out_of_time(dates_left):
    used = (dt.datetime.now(UTC) - RUN_START).total_seconds() / 60
    if used < TIME_BUDGET_MIN:
        return False
    log(f"  time budget reached ({used:.0f} of {TIME_BUDGET_MIN} min): {dates_left} dates left for the next run")
    return True


log(f"{PROJECT_TITLE} | run {RUN_ID} | {ME} @ {gis.url} | mode={RUN_MODE}")

## 2. Helpers

In [ ]:
def to_ms(d):
    if d is None:
        return None
    if isinstance(d, dt.datetime):
        return int((d if d.tzinfo else d.replace(tzinfo=UTC)).timestamp() * 1000)
    return int(dt.datetime(d.year, d.month, d.day, tzinfo=UTC).timestamp() * 1000)


def from_ms(v):
    """Epoch ms, ISO string or datetime -> aware UTC datetime (None stays None)."""
    if v is None or v == "":
        return None
    if isinstance(v, dt.datetime):
        return v if v.tzinfo else v.replace(tzinfo=UTC)
    if isinstance(v, dt.date):
        return dt.datetime(v.year, v.month, v.day, tzinfo=UTC)
    if isinstance(v, (int, float)):
        return dt.datetime.fromtimestamp(v / 1000, UTC)
    s = str(v).replace("Z", "+00:00")
    d = dt.datetime.fromisoformat(s if "T" in s or " " in s else s + "T00:00:00")
    return d if d.tzinfo else d.replace(tzinfo=UTC)


def ts_sql(d):
    return f"timestamp '{d.astimezone(UTC):%Y-%m-%d %H:%M:%S}'"


def rnd(v, n=4):
    if v is None:
        return None
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    return None if math.isnan(f) or math.isinf(f) else round(f, n)


def chunks(seq, n):
    seq = list(seq)
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


def sql_list(values):
    out = []
    for v in values:
        out.append(str(int(v)) if isinstance(v, (int, float)) else "'" + str(v).replace("'", "''") + "'")
    return ",".join(out)


def lower_keys(d):
    return {k.lower(): v for k, v in (d or {}).items()}


def rest(url, params=None, retries=4):
    """POST to an ArcGIS REST endpoint with the portal session; retries with backoff."""
    p = {"f": "json"}
    for k, v in (params or {}).items():
        if v is None:
            continue
        p[k] = json.dumps(v) if isinstance(v, (dict, list)) else v
    last = None
    for attempt in range(retries):
        try:
            res = gis._con.post(url, p)
            if isinstance(res, dict) and "error" in res:
                raise RuntimeError(json.dumps(res["error"])[:500])
            return res
        except Exception as exc:
            last = exc
            if attempt < retries - 1:
                time.sleep(3 * 2 ** attempt)
    raise RuntimeError(f"{url.split('?')[0]} failed: {last}")


_INFO = {}


def layer_info(url):
    if url not in _INFO:
        _INFO[url] = rest(url)
    return _INFO[url]


def oid_field(url):
    info = layer_info(url)
    if info.get("objectIdField"):
        return info["objectIdField"]
    return next(f["name"] for f in info.get("fields", []) if f["type"] == "esriFieldTypeOID")


def query_all(url, where="1=1", fields="*", geometry=False, out_sr=4326, max_offset=None, page=None):
    """All rows of a layer/table, paged by object ID (works on hosted and registered services)."""
    oid = oid_field(url)
    info = layer_info(url)
    page = page or min(int(info.get("maxRecordCount") or 1000), 2000)
    paged = (info.get("advancedQueryCapabilities") or {}).get("supportsPagination", False)
    if fields != "*":
        names = [oid if f.strip().lower() == oid.lower() else f.strip() for f in fields.split(",")]
        fields = ",".join(names if oid in names else [oid] + names)
    rows, last = [], -1
    while True:
        params = {"where": f"({where}) AND {oid} > {last}", "outFields": fields, "orderByFields": f"{oid} ASC",
                  "returnGeometry": "true" if geometry else "false", "outSR": out_sr}
        if paged:
            params["resultRecordCount"] = page
        if max_offset:
            params["maxAllowableOffset"] = max_offset
        feats = rest(url + "/query", params).get("features", [])
        if not feats:
            break
        rows.extend(feats)
        ids = [lower_keys(f["attributes"]).get(oid.lower()) for f in feats]
        last = max(i for i in ids if i is not None)
    return rows


def count_rows(url, where="1=1"):
    return int(rest(url + "/query", {"where": where, "returnCountOnly": "true"}).get("count", 0))


EDIT_ERRORS = []


def apply_edits(url, adds=None, updates=None, deletes=None, batch=1000):
    """Batched applyEdits."""
    if DRY_RUN or not url:
        n = len(adds or []) + len(updates or []) + len(deletes or [])
        if n:
            log(f"  [dry_run] would write {n} rows ({len(adds or [])} adds, {len(updates or [])} updates)")
        return n, []
    ok, errors = 0, []
    fit_lengths(url, (adds or []) + (updates or []))
    oid = oid_field(url) if updates else None
    if updates and oid != "OBJECTID":
        for r in updates:
            a = r["attributes"]
            if "OBJECTID" in a:
                a[oid] = a.pop("OBJECTID")
    for key, rows in (("adds", adds), ("updates", updates)):
        has_geometry = any("geometry" in r for r in (rows or [])[:1])
        for part in chunks(rows or [], max(50, batch // 5) if has_geometry else batch):
            res = rest(url + "/applyEdits", {key: part, "rollbackOnFailure": "false"})
            for r in res.get(key[:-1] + "Results", []):
                if r.get("success"):
                    ok += 1
                else:
                    errors.append(r.get("error"))
    for part in chunks(deletes or [], batch):
        res = rest(url + "/applyEdits", {"deletes": ",".join(str(i) for i in part), "rollbackOnFailure": "false"})
        ok += sum(1 for r in res.get("deleteResults", []) if r.get("success"))
    if errors:
        log(f"  {len(errors)} edit errors, first: {errors[0]}")
        EDIT_ERRORS.extend(errors)
    return ok, errors


def fit_lengths(url, rows):
    """Cut text values to the field lengths of the layer (an over-long value fails the whole row)."""
    try:
        lengths = {f["name"].lower(): f.get("length") for f in layer_info(url).get("fields", [])
                   if f.get("type") == "esriFieldTypeString" and f.get("length")}
    except Exception:
        return
    for r in rows:
        a = r.get("attributes") or {}
        for k, v in a.items():
            n = lengths.get(k.lower())
            if n and isinstance(v, str) and len(v) > n:
                a[k] = v[:n]


def planar_area_m2(geom):
    total = 0.0
    for ring in geom.get("rings", []):
        s = 0.0
        for (x1, y1), (x2, y2) in zip(ring, ring[1:]):
            s += x1 * y2 - x2 * y1
        total += s / 2.0
    return abs(total)


def bbox(geom):
    xs = [p[0] for r in geom.get("rings", []) for p in r]
    ys = [p[1] for r in geom.get("rings", []) for p in r]
    return (min(xs), min(ys), max(xs), max(ys)) if xs else None


def point_in_rings(x, y, rings):
    inside = False
    for ring in rings:
        for (x1, y1), (x2, y2) in zip(ring, ring[1:]):
            if (y1 > y) != (y2 > y) and x < (x2 - x1) * (y - y1) / ((y2 - y1) or 1e-12) + x1:
                inside = not inside
    return inside


def bbox_fields(geom4326):
    b = bbox(geom4326)
    if not b:
        return {}
    return {"cx": rnd((b[0] + b[2]) / 2, 6), "cy": rnd((b[1] + b[3]) / 2, 6),
            "bxmin": rnd(b[0], 6), "bymin": rnd(b[1], 6), "bxmax": rnd(b[2], 6), "bymax": rnd(b[3], 6)}


def sample_points(a):
    """Centre and four inset corners of a unit's bounding box (used for tile-coverage checks)."""
    if a.get("bxmin") is None:
        return [(a.get("cx"), a.get("cy"))]
    dx, dy = (a["bxmax"] - a["bxmin"]) * 0.1, (a["bymax"] - a["bymin"]) * 0.1
    return [(a["cx"], a["cy"]), (a["bxmin"] + dx, a["bymin"] + dy), (a["bxmax"] - dx, a["bymin"] + dy),
            (a["bxmin"] + dx, a["bymax"] - dy), (a["bxmax"] - dx, a["bymax"] - dy)]


def _signed_area(coords):
    return sum(x1 * y2 - x2 * y1 for (x1, y1, *_), (x2, y2, *_) in zip(coords, coords[1:])) / 2.0


def shapely_to_esri(shape):
    polys = list(shape.geoms) if hasattr(shape, "geoms") else [shape]
    rings = []
    for p in polys:
        if p.is_empty or not hasattr(p, "exterior"):
            continue
        ext = list(p.exterior.coords)
        rings.append(ext if _signed_area(ext) < 0 else ext[::-1])
        for hole in p.interiors:
            h = list(hole.coords)
            rings.append(h if _signed_area(h) > 0 else h[::-1])
    return {"rings": [[list(pt[:2]) for pt in r] for r in rings]}


def esri_to_shapely(geom3857):
    from arcgis.geometry import Geometry
    return Geometry(dict(geom3857, spatialReference={"wkid": 3857})).as_shapely


def ensure_folder():
    try:
        gis.content.folders.create(FOLDER, exist_ok=True)
    except Exception:
        try:
            gis.content.create_folder(FOLDER)
        except Exception:
            pass


def share_items(items):
    if DRY_RUN or not SHARE_TO:
        return
    groups = []
    if SHARE_TO not in ("org", "everyone"):
        groups = [g for g in gis.groups.search(f'title:"{SHARE_TO}"') if g.title == SHARE_TO] or \
                 [gis.groups.create(title=SHARE_TO, tags="Rwanda,monitoring", access="org")]
    for it in items:
        try:
            it.share(org=SHARE_TO == "org", everyone=SHARE_TO == "everyone", groups=groups or None)
        except Exception as exc:
            log(f"  could not share {it.title}: {exc}")

## 3. Sentinel-2 Level-2A access
Imagery comes from the **Sentinel2L2A** item in GeoHub. Every index is masked to clear pixels with the scene
classification band (SCL):

| SCL | class | used as |
|---|---|---|
| 4 | vegetation | clear |
| 5 | not vegetated | clear |
| 6 | water | clear |
| 7 | unclassified | clear |
| 3, 8, 9, 10 | cloud shadow, cloud, cirrus | cloudy |
| 0 | no data | ignored |

In [ ]:
S2_BANDS = {"B1": 1, "B2": 2, "B3": 3, "B4": 4, "B5": 5, "B6": 6, "B7": 7, "B8": 8, "B8A": 9, "B9": 10,
            "B11": 11, "B12": 12, "AOT": 13, "WVP": 14, "SCL": 15}
CLEAR_SCL = [4, 5, 6, 7]
RWANDA_ENV = {"xmin": 28.84, "ymin": -2.86, "xmax": 30.91, "ymax": -1.04, "spatialReference": {"wkid": 4326}}
WORK_WKID = 32735
HISTORY_DAYS = int(HISTORY_MONTHS * 30.44)
S2 = {}


def s2_init():
    item = gis.content.get(S2_ITEM_ID)
    if item is None:
        raise RuntimeError(f"Sentinel-2 item {S2_ITEM_ID} not found or not shared with {ME}")
    url = item.url.rstrip("/")
    info = rest(url)
    found = {}
    for pos, name in enumerate(info.get("bandNames") or [], start=1):
        m = re.match(r"^B(\d{1,2}A?)_(.*)$", name)
        if m:
            tail = m.group(2).upper()
            key = "SCL" if "SCL" in tail else "AOT" if "AOT" in tail else "WVP" if "WVP" in tail else "B" + m.group(1)
            found[key] = pos
    needed = ("B2", "B3", "B4", "B5", "B8", "B8A", "B11", "B12", "SCL")
    if all(k in found for k in needed):
        S2_BANDS.update(found)
    elif int(info.get("bandCount") or 0) != 15:
        raise RuntimeError(f"{info.get('name')} does not look like Sentinel-2 L2A (bands: {info.get('bandNames')})")
    else:
        log("WARNING: band names not readable; using the standard 15-band L2A order")
    fields = {f["name"].lower() for f in info.get("fields", [])}
    missing = {"acquisitiondate", "cloudcover", "name", "category"} - fields
    if missing:
        raise RuntimeError(f"Sentinel-2 service is missing fields {missing}")
    S2.update(url=url, item=item, oid=info.get("objectIdField") or "objectid")
    recent = rest(url + "/query", {
        "where": f"category = 1 AND acquisitiondate >= {ts_sql(RUN_START - dt.timedelta(days=60))}",
        "geometry": RWANDA_ENV, "geometryType": "esriGeometryEnvelope", "inSR": 4326,
        "spatialRel": "esriSpatialRelIntersects", "outFields": "cloudcover", "returnGeometry": "false",
        "resultRecordCount": 200})
    cmax = max([lower_keys(f["attributes"]).get("cloudcover") or 0 for f in recent.get("features", [])] + [0])
    S2["cloud_scale"] = 100.0 if cmax > 1.5 else 1.0
    log(f"Sentinel-2: {info.get('name')} | {info.get('bandCount')} bands {info.get('pixelType')} | "
        f"B8A=band {S2_BANDS['B8A']}, B11=band {S2_BANDS['B11']}, SCL=band {S2_BANDS['SCL']} | "
        f"cloudcover in {'percent' if S2['cloud_scale'] > 1 else 'fraction'}")


def find_scenes(start, end, max_cloud=None, env=RWANDA_ENV):
    """Primary Sentinel-2 tiles over the area, with footprints (WGS84)."""
    max_cloud = SCENE_MAX_CLOUD if max_cloud is None else max_cloud
    oid = S2["oid"]
    base = (f"category = 1 AND cloudcover <= {max_cloud * S2['cloud_scale']:g} AND acquisitiondate >= {ts_sql(start)} "
            f"AND acquisitiondate <= {ts_sql(end)}")
    scenes, last = [], -1
    while True:
        res = rest(S2["url"] + "/query", {
            "where": f"{base} AND {oid} > {last}", "geometry": env, "geometryType": "esriGeometryEnvelope",
            "inSR": 4326, "spatialRel": "esriSpatialRelIntersects", "outSR": 4326, "returnGeometry": "true",
            "maxAllowableOffset": 0.002, "outFields": f"{oid},name,acquisitiondate,cloudcover,tilename",
            "orderByFields": f"{oid} ASC"})
        feats = res.get("features", [])
        if not feats:
            break
        for f in feats:
            a = lower_keys(f["attributes"])
            t = from_ms(a["acquisitiondate"])
            scenes.append({"oid": a[oid.lower()], "name": a["name"], "time": t, "day": t.date().isoformat(),
                           "cloud": float(a.get("cloudcover") or 0) / S2["cloud_scale"], "tile": a.get("tilename"),
                           "rings": (f.get("geometry") or {}).get("rings", [])})
        last = max(lower_keys(f["attributes"])[oid.lower()] for f in feats)
    return scenes


def group_by_day(scenes):
    days = {}
    for s in scenes:
        days.setdefault(s["day"], []).append(s)
    return days


def covered(scenes, points, need=0.6):
    """Share of the sample points that fall inside the union of the scene footprints."""
    pts = [p for p in points if p[0] is not None]
    if not pts:
        return False
    hits = sum(1 for x, y in pts if any(point_in_rings(x, y, s["rings"]) for s in scenes))
    return hits / len(pts) >= need


def day_extent(scenes):
    xs = [p[0] for s in scenes for r in s["rings"] for p in r]
    ys = [p[1] for s in scenes for r in s["rings"] for p in r]
    return {"xmin": max(min(xs), RWANDA_ENV["xmin"]), "ymin": max(min(ys), RWANDA_ENV["ymin"]),
            "xmax": min(max(xs), RWANDA_ENV["xmax"]), "ymax": min(max(ys), RWANDA_ENV["ymax"]),
            "spatialReference": {"wkid": 4326}}


def scenelog_done(url, since_day):
    rows = [lower_keys(f["attributes"]) for f in query_all(url, f"acq_day >= '{since_day}'", "scene_name,status")] if url else []
    done = {a["scene_name"] for a in rows if a["status"] in ("processed", "no_units")}
    failures = {}
    for a in rows:
        if a["status"] == "failed":
            failures[a["scene_name"]] = failures.get(a["scene_name"], 0) + 1
    return done | {n for n, c in failures.items() if c >= MAX_SCENE_ATTEMPTS}


def scenelog_write(url, scenes, status, units=0, note=""):
    now = to_ms(dt.datetime.now(UTC))
    rows = [{"attributes": {"scene_name": s["name"], "acq_date": to_ms(s["time"]), "acq_day": s["day"],
                            "tile": s["tile"], "cloud_pct": rnd(100 * s["cloud"], 1), "status": status,
                            "units": units, "note": str(note)[:250], "run_id": RUN_ID, "logged_utc": now}}
            for s in scenes]
    for r in rows:
        r["attributes"]["meaning"] = scene_meaning(r["attributes"])
    if url:
        apply_edits(url, adds=rows)


def scene_meaning(a):
    tile = f"Tile {a.get('tile') or '?'} of {a.get('acq_day')} ({a.get('cloud_pct')}% cloud over the whole tile)"
    return {"processed": f"{tile} was processed: {a.get('units') or 0} monitored units were measured from it and "
                         "their values are in the observations table.",
            "no_units": f"{tile} was checked but covers none of the monitored units, so nothing was measured.",
            "failed": f"{tile} could not be processed ({(a.get('note') or '')[:120]}); it is tried again on the next "
                      f"run, up to {MAX_SCENE_ATTEMPTS} times."}.get(a.get("status"), f"{tile}: {a.get('status')}.")


def pending_days(scenelog_url, priority=None, since=None):
    """Acquisition dates to process in this run, at most MAX_DATES_PER_RUN."""
    first_run = count_rows(scenelog_url) == 0 if scenelog_url else True
    start = RUN_START - dt.timedelta(days=HISTORY_DAYS)
    scenes = find_scenes(start, RUN_START)
    done = scenelog_done(scenelog_url, start.date().isoformat())
    fresh = [s for s in scenes if s["name"] not in done and (since is None or s["day"] >= since)]
    days = group_by_day(fresh)
    only = {d.strip() for d in str(globals().get("ONLY_DATES") or "").split(",") if d.strip()}
    candidates = [d for d in days if not only or d in only]
    score = {d: priority(days[d]) for d in candidates} if priority else {}
    chosen = sorted(candidates, key=lambda d: (score.get(d, 0), d), reverse=True)[:MAX_DATES_PER_RUN]
    gain = sum(1 for d in chosen if score.get(d, 0) > 0)
    log(f"Scenes since {start:%Y-%m-%d} (cloud <= {SCENE_MAX_CLOUD:.0%}): {len(scenes)} | not yet processed: "
        f"{len(fresh)} on {len(days)} dates | this run: {len(chosen)}"
        + (f" ({gain} chosen first because they cover units without a clear image)" if gain else "")
        + (" (first run: back-filling history)" if first_run else ""))
    return {d: days[d] for d in chosen}, len(days) - len(chosen)


def coverage_priority(units, key="last_obs_date"):
    """priority(scenes) for pending_days / day_order: how many units without a clear observation the tiles cover,
    discounted by tile cloud cover."""
    def points():
        return [(u["cx"], u["cy"]) for u in units.values() if not u.get(key) and u.get("cx") is not None]

    def score(scenes, pts=None):
        pts = points() if pts is None else pts
        if not pts:
            return 0
        boxes = []
        for s in scenes:
            xs = [p[0] for r in s["rings"] for p in r]
            ys = [p[1] for r in s["rings"] for p in r]
            if xs:
                boxes.append((min(xs), min(ys), max(xs), max(ys), s["rings"]))
        n = 0
        for x, y in pts:
            for x0, y0, x1, y1, rings in boxes:
                if x0 <= x <= x1 and y0 <= y <= y1 and point_in_rings(x, y, rings):
                    n += 1
                    break
        clear = 1 - sum(s["cloud"] for s in scenes) / max(len(scenes), 1)
        return round(n * clear, 1)
    score.points = points
    return score


def day_order(days, priority=None):
    """Dates to process, best first."""
    left = set(days)
    while left:
        scores = {}
        if priority is not None:
            pts = priority.points()
            if pts:
                scores = {d: priority(days[d], pts) for d in left}
        best = max(left, key=lambda d: (scores.get(d, 0), d))
        left.discard(best)
        yield best


def rf(name, args, pixel_type=None):
    d = {"rasterFunction": name, "rasterFunctionArguments": args}
    if pixel_type:
        d["outputPixelType"] = pixel_type
    return d


def band_ref(b):
    return f"B{S2_BANDS[b]}"


def bandmath(expr):
    return rf("BandArithmetic", {"Method": 0, "BandIndexes": expr}, "F32")


def clamp(raster, lo, hi):
    """Values outside [lo, hi] (and NaN) become NoData."""
    return rf("Mask", {"Raster": raster, "NoDataValues": [], "IncludedRanges": [lo, hi], "NoDataInterpretation": 0}, "F32")


def nd(a, b):
    A, Bb = band_ref(a), band_ref(b)
    return clamp(bandmath(f"({A}-{Bb})/({A}+{Bb}+0.000001)"), -1, 1)


def ratio(a, b, hi=10):
    return clamp(bandmath(f"{band_ref(a)}/({band_ref(b)}+0.000001)"), 0, hi)


def reflectance(b):
    return clamp(bandmath(band_ref(b)), -0.05, 1.5)


def _ranges(classes):
    """Remap ranges [c, c+1) for integer class values."""
    out = []
    for c in classes:
        out += [c, c + 1]
    return out


def scl_band():
    return rf("ExtractBand", {"BandIDs": [S2_BANDS["SCL"] - 1]})


def scl_only(classes):
    """1 where SCL is in `classes`, NoData elsewhere (use as a mask)."""
    return rf("Remap", {"Raster": scl_band(), "InputRanges": _ranges(classes),
                        "OutputValues": [1] * len(classes), "AllowUnmatched": False}, "U8")


def scl_flag(classes, among=CLEAR_SCL):
    """1 where SCL is in `classes`, 0 where it is in the other `among` classes, NoData otherwise."""
    others = [c for c in among if c not in classes]
    return rf("Remap", {"Raster": scl_band(), "InputRanges": _ranges(classes) + _ranges(others),
                        "OutputValues": [1] * len(classes) + [0] * len(others), "AllowUnmatched": False}, "U8")


def clear_flag():
    """1 = clear, 0 = cloud / shadow / cirrus / defective, NoData = outside the image."""
    return scl_flag(CLEAR_SCL, among=list(range(1, 12)))


def threshold01(raster, cut, above=True):
    """1 where raster >= cut (or < cut when above=False), 0 otherwise, NoData stays NoData."""
    return rf("Remap", {"Raster": raster, "InputRanges": [cut, 1e9, -1e9, cut],
                        "OutputValues": [1, 0] if above else [0, 1], "AllowUnmatched": False}, "F32")


def times(r1, r2):
    return rf("Arithmetic", {"Raster": r1, "Raster2": r2, "Operation": 3, "ExtentType": 1, "CellsizeType": 1}, "F32")


def masked(raster, mask):
    return times(raster, mask)


def in_range_only(raster, lo, hi):
    """1 where lo <= raster < hi, NoData elsewhere (a mask that hides everything else)."""
    return rf("Remap", {"Raster": raster, "InputRanges": [lo, hi], "OutputValues": [1],
                        "AllowUnmatched": False}, "U8")


def class_colormap(index_raster, breaks, colors, mask=None):
    """Classified colour overlay for a web map: `breaks` are the class edges (n+1 values), `colors` are n hex colours
    from the low to the high class."""
    ranges = [v for i in range(len(breaks) - 1) for v in (breaks[i], breaks[i + 1])]
    args = {"InputRanges": ranges, "OutputValues": list(range(1, len(breaks))), "AllowUnmatched": False}
    if index_raster is not None:
        args["Raster"] = index_raster
    classes = rf("Remap", args, "U8")
    if mask is not None:
        classes = rf("Arithmetic", {"Raster": classes, "Raster2": mask, "Operation": 3,
                                    "ExtentType": 1, "CellsizeType": 1}, "U8")
    table = [[i + 1] + hexrgb(c) for i, c in enumerate(colors)]
    return rf("Colormap", {"Raster": classes, "Colormap": table})


def hexrgb(h):
    h = h.lstrip("#")
    return [int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)]


def interp_palette(colors, n):
    """`n` colours spread evenly through `colors`, interpolated in RGB."""
    stops = [hexrgb(c) for c in colors]
    if n <= len(stops):
        return list(colors)
    out = []
    for i in range(n):
        t = i / (n - 1) * (len(stops) - 1)
        k = min(int(t), len(stops) - 2)
        f = t - k
        out.append("#%02x%02x%02x" % tuple(
            int(round(stops[k][c] + (stops[k + 1][c] - stops[k][c]) * f)) for c in range(3)))
    return out


def ramp_colormap(index_raster, lo, hi, colors, steps=64, mask=None):
    """A smooth colour ramp across [lo, hi] instead of named classes."""
    edges = [lo + (hi - lo) * i / steps for i in range(steps + 1)]
    edges[0], edges[-1] = -1e9, 1e9
    return class_colormap(index_raster, edges, interp_palette(colors, steps), mask)


def composite(rasters):
    return rf("CompositeBand", {"Rasters": rasters}, "F32")


def scl01(classes, universe=range(1, 12)):
    """1 where SCL is in `classes`, 0 for the other valid classes, NoData only where SCL = 0 (outside the image)."""
    others = [c for c in universe if c not in classes]
    return rf("Remap", {"Raster": scl_band(), "InputRanges": _ranges(classes) + _ranges(others),
                        "OutputValues": [1] * len(classes) + [0] * len(others), "AllowUnmatched": False}, "F32")


def in_range01(raster, lo, hi):
    return rf("Remap", {"Raster": raster, "InputRanges": [-1e9, lo, lo, hi, hi, 1e9],
                        "OutputValues": [0, 1, 0], "AllowUnmatched": False}, "F32")


def nd_raw(a, b):
    A, Bb = band_ref(a), band_ref(b)
    return bandmath(f"({A}-{Bb})/({A}+{Bb}+0.000001)")


def ratio_raw(a, b):
    return bandmath(f"{band_ref(a)}/({band_ref(b)}+0.000001)")


def product(*rasters):
    out = rasters[0]
    for r in rasters[1:]:
        out = times(out, r)
    return out


def wmean(stats, value_idx, weight_idx, digits=5):
    w = stats[weight_idx].get("mean") or 0
    return rnd(stats[value_idx].get("mean") / w, digits) if w > 0 and stats[value_idx].get("mean") is not None else None


def lock_rule(oids):
    ids = sorted(int(i) for i in oids)
    return {"mosaicMethod": "esriMosaicLockRaster", "lockRasterIds": ids, "ascending": True,
            "mosaicOperation": "MT_FIRST", "where": f"{S2['oid']} IN ({sql_list(ids)})"}


def s2_token():
    try:
        return gis._con._create_token(S2["url"])
    except Exception:
        return getattr(gis._con, "token", None)


def ra_s2_input(rule, oids):
    """Raster analytics input: the Sentinel-2 service, locked to one date's tiles, with a function chain."""
    tok = s2_token()
    return {"url": S2["url"] + (f"?token={tok}" if tok else ""), "mosaicRule": lock_rule(oids), "renderingRule": rule}


def s2_live_mosaic(days=90, by="date"):
    """Recent tiles for the web map's live imagery layers: `by="date"` takes the newest, `by="clear"` the least
    cloudy of the last `days` (better for index overlays, whose cloud mask would otherwise leave holes)."""
    since = (RUN_START - dt.timedelta(days=days)).strftime("%Y-%m-%d")
    order = {"sortField": "acquisitiondate", "sortValue": RUN_START.strftime("%Y/%m/%d")} if by == "date" else \
            {"sortField": "cloudcover", "sortValue": "0"}
    return {"mosaicMethod": "esriMosaicAttribute", "ascending": True, "mosaicOperation": "MT_FIRST", **order,
            "where": (f"category = 1 AND cloudcover <= {SCENE_MAX_CLOUD * S2['cloud_scale']:g} "
                      f"AND acquisitiondate >= timestamp '{since} 00:00:00'")}

## 4. Statistics engines

In [ ]:
def image_stats(geometry, oids, rule, pixel_m):
    res = rest(S2["url"] + "/computeStatisticsHistograms", {
        "geometry": geometry, "geometryType": "esriGeometryPolygon", "mosaicRule": lock_rule(oids),
        "renderingRule": rule, "pixelSize": {"x": pixel_m, "y": pixel_m, "spatialReference": {"wkid": 3857}}})
    return [lower_keys(s) for s in res.get("statistics") or []]


def run_image_stats(tasks, rule, workers):
    """tasks: [(key, geometry_3857, oids, pixel_m)] -> {key: [band stats] | {"error": msg}}"""
    def one(task):
        key, geometry, oids, pixel = task
        try:
            return key, image_stats(geometry, oids, rule, pixel)
        except Exception as exc:
            return key, {"error": str(exc)[:300]}

    out = {}
    with ThreadPoolExecutor(max(1, workers)) as pool:
        for key, res in pool.map(one, tasks):
            out[key] = res
    return out


def stats_pixel(geom3857, base_m, max_pixels=1_500_000):
    """Pixel size for one request: at least base_m, under the 5000 x 5000 image limit, and coarse enough that very
    large polygons (Lake Kivu) stay around max_pixels."""
    b = bbox(geom3857)
    if not b:
        return base_m
    by_extent = math.ceil(max(b[2] - b[0], b[3] - b[1]) / 4500.0)
    by_count = math.ceil(math.sqrt(planar_area_m2(geom3857) / max_pixels))
    return max(base_m, by_extent, by_count)


def stats_pieces(geom3857, base_m, max_px=2_000_000):
    """One statistics request per unit, or several pieces for a unit whose bounding box is mostly land."""
    px = stats_pixel(geom3857, base_m)
    b = bbox(geom3857)
    if not b:
        return [(geom3857, base_m)]
    wide = (b[2] - b[0]) / px * (b[3] - b[1]) / px
    if wide <= max_px:
        return [(geom3857, px)]
    try:
        from shapely.geometry import box as shapely_box
        shape = esri_to_shapely(geom3857).buffer(0)
    except Exception as exc:
        log(f"  could not split the unit into pieces ({exc}); using {px} m pixels")
        return [(geom3857, px)]
    n = max(1, math.ceil(math.sqrt(wide / max_px)))
    dx, dy = (b[2] - b[0]) / n, (b[3] - b[1]) / n
    pieces = []
    for i in range(n):
        for j in range(n):
            cell = shapely_box(b[0] + i * dx, b[1] + j * dy, b[0] + (i + 1) * dx, b[1] + (j + 1) * dy)
            part = shape.intersection(cell)
            if not part.is_empty and part.area > 0:
                pieces.append((shapely_to_esri(part), px))
    return pieces or [(geom3857, px)]


def merge_stats(pieces):
    """Band statistics of several pieces of one unit -> the statistics of the whole, weighted by pixel count."""
    good = [p for p in pieces if isinstance(p, list) and p]
    if not good or len(good) != len(pieces):
        return None
    bands = min(len(p) for p in good)
    out = []
    for i in range(bands):
        counts = [(p[i].get("count") or 0, p[i].get("mean") or 0.0) for p in good]
        total = sum(c for c, _ in counts)
        out.append({"count": total,
                    "mean": (sum(c * m for c, m in counts) / total) if total else 0.0,
                    "min": min((p[i].get("min") for p in good if p[i].get("min") is not None), default=None),
                    "max": max((p[i].get("max") for p in good if p[i].get("max") is not None), default=None)})
    return out


def delete_named(title):
    for it in gis.content.search(f'title:"{title}" AND owner:{ME}', max_items=20):
        if it.title == title:
            try:
                it.delete()
            except Exception:
                try:
                    it.delete(force=True)
                except Exception as exc:
                    log(f"  could not delete {title}: {exc}")


def ra_table_rows(item):
    layer = (item.tables or item.layers)[0]
    return [lower_keys(f["attributes"]) for f in query_all(layer.url)]


def job_error(job):
    """Error causes reported by a failed raster analytics job."""
    for obj in (job, getattr(job, "_gpjob", None)):
        try:
            errs = [m.get("description", "") for m in (obj.messages or []) if m.get("type") == "esriJobMessageTypeError"]
        except Exception:
            continue
        causes = re.findall(r'Cause of failure: ([^"]{0,160})', " ".join(errs))
        return "; ".join(dict.fromkeys(causes)) or " | ".join(errs)[:300]
    return ""


def ra_zonal(zone_layer, bands, oids, extent, cell_m, label, attempts=3):
    """bands: [(name, rule, statistic_type)]."""
    from arcgis.raster.analytics import zonal_statistics_as_table
    if DRY_RUN:
        log(f"  [dry_run] would run {len(bands)} zonal statistics jobs for {label}")
        return {}
    ctx = {"cellSize": cell_m, "outSR": {"wkid": WORK_WKID}, "extent": extent, "resamplingMethod": "NEAREST"}
    results, todo, errors = {}, list(bands), []
    for attempt in range(1, attempts + 1):
        jobs = {}
        for name, rule, stat in todo:
            out_name = f"tmp_{PROJECT_CODE}_{label}_{name}_{RUN_ID[:6]}" + (f"_r{attempt}" if attempt > 1 else "")
            try:
                jobs[name] = (zonal_statistics_as_table(
                    input_zone_raster_or_features=zone_layer, input_value_raster=ra_s2_input(rule, oids),
                    zone_field="Value", ignore_nodata=True, statistic_type=stat, output_name=out_name,
                    context=ctx, gis=gis, future=True), out_name)
            except Exception as exc:
                errors.append(f"{name}: submit failed: {exc}")
        errors = []
        for name, (job, out_name) in jobs.items():
            try:
                rows = ra_table_rows(job.result())
                if not results:
                    log(f"  zonal table fields: {sorted(rows[0].keys()) if rows else 'empty'}")
                zone_key = next((k for k in ("value", "zone", "zone_code") if rows and k in rows[0]), "value")
                results[name] = {int(r[zone_key]): r for r in rows if r.get(zone_key) is not None}
            except Exception as exc:
                errors.append(f"{name}: {job_error(job) or exc}")
            finally:
                if not KEEP_INTERMEDIATE:
                    delete_named(out_name)
        todo = [b for b in bands if b[0] not in results]
        if not todo:
            break
        if attempt < attempts:
            log(f"  {label}: {len(todo)} zonal job(s) failed ({'; '.join(errors)[:300]}); retrying")
            time.sleep(60 * attempt)
    if todo:
        raise RuntimeError("zonal statistics failed: " + "; ".join(errors)[:1500])
    return results


TRANSIENT_RA = ("DBMS error", "Underlying DBMS", "timed out", "Timeout", "503", "502", "Connection")


def ra_persist(raster_input, name, cell_m, extent=None, resampling="NEAREST", attempts=3):
    """Save a raster as a hosted imagery layer; retries when the imagery source reports a transient error."""
    for attempt in range(1, attempts + 1):
        try:
            return _ra_persist(raster_input, name if attempt == 1 else f"{name}_r{attempt}", cell_m, extent, resampling)
        except Exception as exc:
            text = str(exc)
            if attempt == attempts or not any(t in text for t in TRANSIENT_RA):
                raise
            log(f"  {name}: transient raster analytics error, retrying ({attempt}/{attempts - 1})")
            delete_named(name if attempt == 1 else f"{name}_r{attempt}")
            time.sleep(60 * attempt)


def _ra_persist(raster_input, name, cell_m, extent=None, resampling="NEAREST"):
    """Save a raster (dict input or ImageryLayer chain) as a hosted imagery layer (CRF in the raster store)."""
    ctx = {"cellSize": cell_m, "outSR": {"wkid": WORK_WKID}, "extent": extent or RWANDA_ENV,
           "resamplingMethod": resampling}
    if DRY_RUN:
        log(f"  [dry_run] would save raster {name}")
        return None
    if isinstance(raster_input, dict):
        from arcgis.raster.analytics import copy_raster
        item = copy_raster(input_raster=raster_input, output_name=name, context=ctx, gis=gis)
    else:
        item = raster_input.save(output_name=name, context=ctx, gis=gis)
    if not hasattr(item, "itemid"):
        item = next(i for i in gis.content.search(f'title:"{name}" AND owner:{ME}', max_items=10) if i.title == name)
    try:
        item.move(FOLDER)
    except Exception:
        pass
    return item


def ra_job(task, params, result, timeout_min=180):
    """Run a raster analytics task over REST and return its `result` value (dict)."""
    ra = gis.properties["helperServices"]["rasterAnalytics"]["url"].rstrip("/")
    job = rest(f"{ra}/{task}/submitJob", params)
    job_url = f"{ra}/jobs/{job['jobId']}"
    deadline = time.time() + timeout_min * 60
    status = job.get("jobStatus")
    while status not in ("esriJobSucceeded", "esriJobFailed", "esriJobCancelled", "esriJobTimedOut"):
        if time.time() > deadline:
            raise RuntimeError(f"{task} did not finish in {timeout_min} min ({job_url})")
        time.sleep(15)
        status = rest(job_url).get("jobStatus")
    if status != "esriJobSucceeded":
        msgs = [m.get("description", "") for m in rest(job_url).get("messages", []) if m.get("type") == "esriJobMessageTypeError"]
        raise RuntimeError(f"{task}: {status} {' | '.join(msgs)[:600]}")
    value = rest(f"{job_url}/results/{result}").get("value") or {}
    return json.loads(value) if isinstance(value, str) else value


def ra_output_item(value, name):
    item = gis.content.get(value["itemId"]) if value.get("itemId") else None
    if item is None:
        item = next(i for i in gis.content.search(f'title:"{name}" AND owner:{ME}', max_items=10) if i.title == name)
    try:
        item.move(FOLDER)
    except Exception:
        pass
    return item


def check_cell(item, cell_m):
    cell = (rest(item.url) or {}).get("pixelSizeX")
    if cell is None or abs(float(cell) - cell_m) > cell_m:
        raise RuntimeError(f"{item.title} has {cell} m cells instead of {cell_m} m")
    log(f"  {item.title}: {float(cell):.2f} m cells")


def raster_has_pixels(item):
    """Does a saved raster actually contain data?"""
    if item is None:
        return False
    url = item.layers[0].url
    try:
        stats = rest(url + "/computeStatisticsHistograms", {
            "geometry": {"xmin": RWANDA_ENV["xmin"], "ymin": RWANDA_ENV["ymin"], "xmax": RWANDA_ENV["xmax"],
                         "ymax": RWANDA_ENV["ymax"], "spatialReference": {"wkid": 4326}},
            "geometryType": "esriGeometryEnvelope",
            "pixelSize": {"x": 200, "y": 200, "spatialReference": {"wkid": WORK_WKID}}})
        count = ((stats.get("statistics") or [{}])[0] or {}).get("count")
    except Exception as exc:
        log(f"  could not check the overlay raster ({exc})")
        return True
    log(f"  overlay raster: {count or 0:,} pixels sampled at 200 m")
    return bool(count)


def ra_upload(path):
    """Upload a local raster file to the raster analytics server; returns the upload's item ID."""
    ra = gis.properties["helperServices"]["rasterAnalytics"]["url"].rstrip("/")
    res = gis._con.post(path=f"{ra}/uploads/upload", postdata={"f": "json", "filename": os.path.basename(path),
                                                               "overwrite": True},
                        files={"file": path}, timeout=None)
    if not isinstance(res, dict) or "item" not in res:
        raise RuntimeError(f"upload of {os.path.basename(path)} failed: {str(res)[:300]}")
    return res["item"]["itemID"]


def ra_delete_upload(upload_id):
    ra = gis.properties["helperServices"]["rasterAnalytics"]["url"].rstrip("/")
    try:
        rest(f"{ra}/uploads/{upload_id}/delete")
    except Exception as exc:
        log(f"  could not delete upload {upload_id}: {exc}")


def ra_publish_files(paths, name, cell_m, timeout_min=120):
    """Local GeoTIFFs -> one hosted imagery layer (Copy Raster mosaics them)."""
    if DRY_RUN:
        log(f"  [dry_run] would publish {len(paths)} files as {name}")
        return None
    ids = []
    try:
        for path in paths:
            ids.append(ra_upload(path))
        value = ra_job("CopyRaster", {
            "inputRaster": {"itemIds": ids, "itemsOnServer": True,
                            "rasterType": {"rasterTypeName": "Raster Dataset", "rasterTypeParameters": None}},
            "outputName": {"serviceProperties": {"name": name}},
            "context": {"outSR": {"wkid": WORK_WKID}}}, "outputRaster", timeout_min)
    finally:
        for upload_id in ids:
            ra_delete_upload(upload_id)
    item = ra_output_item(value, name)
    check_cell(item, cell_m)
    return item


def feature_input(layer_url, where=None):
    d = {"url": layer_url, "serviceToken": gis._con.token}
    if where:
        d["filter"] = where
    return d


def ra_feature_to_raster(layer_url, value_field, name, cell_m, timeout_min=180):
    """Polygons -> integer zone raster (hosted imagery layer) at cell_m metres in UTM 35S."""
    if DRY_RUN:
        log(f"  [dry_run] would rasterise {layer_url.rsplit('/', 2)[-2]} -> {name}")
        return None
    box = (rest(layer_url + "/query", {"where": "1=1", "returnExtentOnly": "true",
                                       "outSR": WORK_WKID}) or {}).get("extent") or {}
    pad = 10 * cell_m
    extent = ({"xmin": box["xmin"] - pad, "ymin": box["ymin"] - pad, "xmax": box["xmax"] + pad,
               "ymax": box["ymax"] + pad, "spatialReference": {"wkid": WORK_WKID}}
              if {"xmin", "ymin", "xmax", "ymax"} <= set(box) else RWANDA_ENV)
    value = ra_job("ConvertFeatureToRaster", {
        "inputFeature": feature_input(layer_url),
        "outputName": {"serviceProperties": {"name": name}},
        "outputCellSize": {"distance": cell_m, "units": "esriMeters"},
        "valueField": value_field,
        "context": {"outSR": {"wkid": WORK_WKID}, "extent": extent,
                    "cellSize": {"x": cell_m, "y": cell_m, "spatialReference": {"wkid": WORK_WKID}}}},
        "outputRaster", timeout_min)
    item = ra_output_item(value, name)
    check_cell(item, cell_m)
    return item


def ra_distance(layer_url, name, cell_m, max_m, where=None, timeout_min=180):
    """Straight-line distance (m) to the features of a layer, capped at max_m, as a hosted imagery layer."""
    if DRY_RUN:
        log(f"  [dry_run] would compute distance to {layer_url.rsplit('/', 2)[-2]} -> {name}")
        return None
    value = ra_job("CalculateDistance", {
        "inputSourceRasterOrFeatures": feature_input(layer_url, where),
        "outputDistanceName": {"serviceProperties": {"name": name}},
        "maximumDistance": {"distance": max_m, "units": "esriMeters"},
        "outputCellSize": {"distance": cell_m, "units": "esriMeters"},
        "distanceMethod": "PLANAR",
        "context": {"outSR": {"wkid": WORK_WKID}, "extent": RWANDA_ENV}}, "outputDistanceRaster", timeout_min)
    item = ra_output_item(value, name)
    check_cell(item, cell_m)
    return item


def ra_raster_to_polygons(raster, name, timeout_min=120):
    """Integer raster (url, or {"url", "renderingRule"}) -> hosted polygon layer (field gridcode), via REST."""
    value = ra_job("ConvertRasterToFeature", {
        "inputRaster": raster if isinstance(raster, dict) else {"url": raster},
        "outputName": {"serviceProperties": {"name": name}},
        "field": "Value", "outputType": "Polygon", "simplifyLinesOrPolygons": True,
        "createMultipartFeatures": False,
        "context": {"outSR": {"wkid": 4326}}}, "outputFeatures", timeout_min)
    return ra_output_item(value, name)

## 5. History analytics (last 14 months)
Baseline, recent reference and percentile rank, computed from each
unit's monthly history (`hist_json`).

In [ ]:
def month_key(day):
    return f"{day.year:04d}-{day.month:02d}"


def month_index(key):
    y, m = key.split("-")
    return int(y) * 12 + int(m) - 1


def hist_load(text):
    try:
        return json.loads(text) if text else {}
    except ValueError:
        return {}


def hist_add(hist, day, values):
    month = hist.setdefault(month_key(day), {})
    for k, v in values.items():
        if v is None:
            continue
        mean, n = month.get(k, [0.0, 0])
        month[k] = [round((mean * n + float(v)) / (n + 1), 5), n + 1]


def hist_dump(hist, keep_months=None, max_len=3900):
    keep = keep_months or HISTORY_MONTHS
    newest = month_index(month_key(TODAY))
    h = {k: v for k, v in hist.items() if newest - month_index(k) < keep}
    text = json.dumps(dict(sorted(h.items())), separators=(",", ":"))
    while len(text) > max_len and h:
        h.pop(min(h))
        text = json.dumps(dict(sorted(h.items())), separators=(",", ":"))
    return text


def hist_series(hist, key, exclude=None):
    out = []
    for m in sorted(hist):
        if m == exclude or key not in hist[m]:
            continue
        out.append((month_index(m), hist[m][key][0]))
    return out


def hist_baseline(hist, key, exclude=None):
    vals = [v for _, v in hist_series(hist, key, exclude)]
    return sum(vals) / len(vals) if vals else None


def hist_recent(hist, key, day, months=3):
    """Mean of the `months` monthly means before the month of `day`."""
    cur = month_index(month_key(day))
    vals = [v for i, v in hist_series(hist, key) if 1 <= cur - i <= months]
    return sum(vals) / len(vals) if vals else None


def hist_count(hist):
    return sum(max((v[1] for v in m.values()), default=0) for m in hist.values())


def pct_rank(hist, key, value):
    vals = [v for _, v in hist_series(hist, key)]
    if value is None or len(vals) < 3:
        return None
    return round(100.0 * sum(1 for v in vals if v <= value) / len(vals), 1)


def fmt_date(v):
    d = from_ms(v)
    return f"{d.day} {d:%b %Y}" if d else "never"


def ago(days):
    if days is None:
        return ""
    return "today" if days == 0 else "yesterday" if days == 1 else f"{days} days ago"

## 6. Hosted feature service, admin boundaries and logs

In [ ]:
FT = {"str": "esriFieldTypeString", "int": "esriFieldTypeInteger", "sint": "esriFieldTypeSmallInteger",
      "dbl": "esriFieldTypeDouble", "date": "esriFieldTypeDate"}


def F(name, kind, length=None, alias=None, domain=None):
    d = {"name": name, "type": FT[kind], "alias": alias or name.replace("_", " ").capitalize(),
         "nullable": True, "editable": True}
    if kind == "str":
        d["length"] = length or 64
    if domain:
        d["domain"] = {"type": "codedValue", "name": f"{PROJECT_CODE}_{name}",
                       "codedValues": [{"name": v, "code": v} for v in domain]}
    return d


ADMIN_FIELDS = [F("province", "str", 50), F("province_id", "str", 2), F("district", "str", 50),
                F("district_id", "str", 4), F("sector", "str", 60), F("sector_id", "str", 6),
                F("sector_label", "str", 120, "Sector (district)"),
                F("cell", "str", 60), F("cell_id", "str", 8), F("village", "str", 60),
                F("village_id", "str", 10), F("village_label", "str", 130, "Village (cell)")]
BBOX_FIELDS = [F("cx", "dbl", alias="Centre lon"), F("cy", "dbl", alias="Centre lat"), F("bxmin", "dbl"),
               F("bymin", "dbl"), F("bxmax", "dbl"), F("bymax", "dbl")]
RUN_STATUSES = ["success", "no_new_data", "partial", "failed", "skipped", "running"]
MEANING = F("meaning", "str", 800, "What this row means")
COMMON_TABLES = [
    {"name": "IndexGuide", "fields": [F("code", "str", 24), F("name", "str", 100), F("formula", "str", 200),
                                      F("bands", "str", 80), F("meaning", "str", 600), F("how_to_read", "str", 800),
                                      F("thresholds", "str", 500), F("caveats", "str", 600),
                                      F("reference", "str", 300), F("sort_order", "sint")]},
    {"name": "SceneLog", "indexes": ["scene_name", "acq_day"],
     "fields": [F("scene_name", "str", 100), F("acq_date", "date", alias="Acquired (UTC)"), F("acq_day", "str", 10),
                F("tile", "str", 12), F("cloud_pct", "dbl", alias="Tile cloud %"), F("status", "str", 16),
                F("units", "int", alias="Units observed"), F("note", "str", 255), F("run_id", "str", 16),
                F("logged_utc", "date"), MEANING]},
    {"name": "RunLog", "indexes": ["finished_utc"],
     "fields": [F("run_id", "str", 16), F("stage", "str", 24), F("status", "str", 16, domain=RUN_STATUSES),
                F("run_mode", "str", 16), F("started_utc", "date"), F("finished_utc", "date"),
                F("dates", "int", alias="Image dates"), F("scenes", "int", alias="Tiles"), F("units", "int"),
                F("observations", "int"), F("alerts", "int"), F("latest_image", "date"),
                F("message", "str", 4000), MEANING]},
    {"name": "RasterRegistry", "indexes": ["kind"],
     "fields": [F("kind", "str", 24), F("label", "str", 60), F("season", "str", 8), F("acq_day", "str", 10),
                F("item_id", "str", 40), F("url", "str", 400), F("title", "str", 200), F("cell_m", "dbl"),
                F("active", "sint"), F("created_utc", "date"), F("note", "str", 400), MEANING]},
]
PARTS_ID = {}
OID_FIELD = {"name": "OBJECTID", "type": "esriFieldTypeOID", "alias": "OBJECTID", "nullable": False, "editable": False}


def unique_domains(spec, fields):
    """Domain names must be unique per service, so prefix them with the layer name."""
    out = []
    for f in fields:
        if f.get("domain"):
            f = dict(f, domain=dict(f["domain"], name=f"{PROJECT_CODE}_{spec['name']}_{f['name']}"))
        out.append(f)
    return out


def layer_json(idx, spec):
    d = {"id": idx, "name": spec["name"], "type": "Feature Layer" if spec.get("geometry") else "Table",
         "objectIdField": "OBJECTID", "fields": [OID_FIELD] + unique_domains(spec, spec["fields"]),
         "hasAttachments": False,
         "displayField": spec.get("display", spec["fields"][0]["name"]),
         "capabilities": "Query,Create,Update,Delete,Editing,Extract",
         "indexes": [{"name": f"ix_{spec['name'][:14]}_{f}"[:30], "fields": f, "isAscending": True,
                      "isUnique": False, "description": ""} for f in spec.get("indexes", [])]}
    if spec.get("geometry"):
        d.update({"geometryType": spec["geometry"], "extent": RWANDA_ENV,
                  "drawingInfo": {"renderer": spec.get("renderer") or {
                      "type": "simple", "symbol": {"type": "esriSFS", "style": "esriSFSSolid", "color": [120, 144, 156, 90],
                                                   "outline": {"type": "esriSLS", "style": "esriSLSSolid",
                                                               "color": [60, 60, 60, 200], "width": 0.5}}}}})
    return d


def all_specs():
    names = {s["name"] for s in SCHEMA["tables"]}
    return SCHEMA["layers"], SCHEMA["tables"] + [t for t in COMMON_TABLES if t["name"] not in names]


def find_item(title, item_type=None):
    for it in gis.content.search(f'title:"{title}" AND owner:{ME}', max_items=50):
        if it.title == title and (item_type is None or it.type == item_type):
            return it
    return None


def sync_schema(item):
    layers, tables = all_specs()
    flc = FeatureLayerCollection.fromitem(item)
    existing = {l.properties.name: l for l in list(item.layers) + list(item.tables)}
    next_id = max([l.properties.id for l in existing.values()] + [-1]) + 1
    new_layers, new_tables = [], []
    for spec, bucket in [(s, new_layers) for s in layers] + [(s, new_tables) for s in tables]:
        lyr = existing.get(spec["name"])
        if lyr is None:
            bucket.append(layer_json(next_id, spec))
            next_id += 1
            continue
        have = {f["name"].lower(): f for f in lyr.properties.fields}
        missing = [f for f in spec["fields"] if f["name"].lower() not in have]
        if missing:
            log(f"  adding {len(missing)} field(s) to {spec['name']}: {[f['name'] for f in missing]}")
            lyr.manager.add_to_definition({"fields": unique_domains(spec, missing)})
        grown = []
        for f in spec["fields"]:
            old = have.get(f["name"].lower())
            want = [c["code"] for c in (f.get("domain") or {}).get("codedValues", [])]
            got = [c["code"] for c in ((dict(old).get("domain") or {}).get("codedValues") or [])] if old else []
            if old and want and set(want) - set(got):
                grown.append(dict(unique_domains(spec, [f])[0], name=old["name"]))
        if grown:
            log(f"  extending value lists in {spec['name']}: {[f['name'] for f in grown]}")
            lyr.manager.update_definition({"fields": grown})
    if new_layers or new_tables:
        log(f"  adding {[d['name'] for d in new_layers + new_tables]} to {SERVICE_NAME}")
        flc.manager.add_to_definition({"layers": new_layers, "tables": new_tables})


def item_folder(item):
    """Title of the folder an item is in (None for the root folder or if it cannot be read)."""
    fid = getattr(item, "ownerFolder", None)
    if not fid:
        return None
    try:
        res = rest(f"{gis._portal.resturl.rstrip('/')}/content/users/{ME}", {"num": 1})
    except Exception:
        return None
    return next((f["title"] for f in res.get("folders", []) if f["id"] == fid), None)


def ensure_service():
    """-> (item, {layer/table name: url}). Creates the service on the first run."""
    global FOLDER
    layers, tables = all_specs()
    item = find_item(SERVICE_NAME, "Feature Service")
    if item is None and DRY_RUN:
        log(f"[dry_run] {SERVICE_NAME} does not exist yet; continuing without stored state")
        return None, {s["name"]: None for s in layers + tables}
    moved = item_folder(item) if item is not None else None
    if moved and moved != FOLDER:
        log(f"  the service is in the folder '{moved}'; new items are saved there")
        FOLDER = moved
    ensure_folder()
    if item is None:
        log(f"Creating hosted feature service {SERVICE_NAME} ...")
        item = gis.content.create_service(
            name=SERVICE_NAME, service_type="featureService", has_static_data=False, max_record_count=2000,
            wkid=4326, folder=FOLDER,
            create_params={"name": SERVICE_NAME, "maxRecordCount": 2000, "spatialReference": {"wkid": 4326},
                           "capabilities": "Query,Create,Update,Delete,Editing,Extract",
                           "initialExtent": RWANDA_ENV, "fullExtent": RWANDA_ENV})
        defs = [layer_json(i, s) for i, s in enumerate(layers)]
        tdefs = [layer_json(len(layers) + i, s) for i, s in enumerate(tables)]
        FeatureLayerCollection.fromitem(item).manager.add_to_definition({"layers": defs, "tables": tdefs})
        item.update(item_properties={"snippet": SERVICE_SNIPPET, "description": SERVICE_DESCRIPTION, "tags": TAGS})
        item = gis.content.get(item.id)
    elif not DRY_RUN:
        sync_schema(item)
        item = gis.content.get(item.id)
    parts = {l.properties.name: l.url for l in list(item.layers) + list(item.tables)}
    ids = {l.properties.name: l.properties.id for l in list(item.layers) + list(item.tables)}
    PARTS_ID.update(ids)
    log(f"Service {SERVICE_NAME} ({item.id}): {', '.join(f'{k}={v}' for k, v in ids.items())}")
    return item, parts


def create_view_rest(item, name):
    """Create the query-only view through the REST API."""
    res = rest(f"{gis._portal.resturl.rstrip('/')}/content/users/{ME}/createService",
               {"outputType": "featureService", "title": name, "tags": TAGS, "snippet": SERVICE_SNIPPET,
                "createParameters": {"name": name, "isView": True, "sourceSchemaChangesAllowed": True,
                                     "isUpdatableView": False, "capabilities": "Query",
                                     "spatialReference": {"wkid": 4326}, "initialExtent": RWANDA_ENV,
                                     "fullExtent": RWANDA_ENV, "serviceDescription": SERVICE_SNIPPET}})
    if not res.get("itemId"):
        raise RuntimeError(f"could not create the view: {str(res)[:300]}")
    view = gis.content.get(res["itemId"])
    sync_view(view, item)
    return view


def ensure_view(item):
    name = SERVICE_NAME + "_View"
    view = find_item(name, "Feature Service")
    if view is None and item is not None and not DRY_RUN:
        log(f"Creating read-only view {name}")
        mgr = FeatureLayerCollection.fromitem(item).manager
        try:
            view = mgr.create_view(name=name, capabilities="Query", folder=FOLDER)
        except Exception:
            view = create_view_rest(item, name)
        try:
            view.move(FOLDER)
        except Exception:
            pass
        view.update(item_properties={"snippet": "Read-only view used by the web map and dashboard. " + SERVICE_SNIPPET,
                                     "tags": TAGS})
    elif view is not None and item is not None and not DRY_RUN:
        sync_view(view, item)
    return view


def ensure_edit_view(item, layers=("FieldChecks",), suffix="_FieldEdit",
                     capabilities="Query,Create,Update,Delete,Editing"):
    """A view that holds only `layers` and allows editing, so users can draw fields without being able to change the
    monitoring layers."""
    name = SERVICE_NAME + suffix
    view = find_item(name, "Feature Service")
    if view is not None:
        sync_view_fields(view, item, layers)
        return view
    if item is None or DRY_RUN:
        return view
    log(f"Creating editable view {name} ({', '.join(layers)})")
    mgr = FeatureLayerCollection.fromitem(item).manager
    try:
        view = mgr.create_view(name=name, capabilities=capabilities, folder=FOLDER)
    except TypeError:
        view = mgr.create_view(name=name, capabilities=capabilities)
    admin = view.url.rstrip("/").replace("/rest/services/", "/rest/admin/services/")
    drop = [{"id": l.properties.id} for l in list(view.layers) + list(view.tables)
            if l.properties.name not in layers]
    if drop:
        rest(admin + "/deleteFromDefinition", {"deleteFromDefinition": {"layers": drop}})
    rest(admin + f"/{PARTS_ID[layers[0]]}/updateDefinition",
         {"updateDefinition": {"capabilities": capabilities, "allowGeometryUpdates": True}})
    view.update(item_properties={
        "snippet": "Draw a field here to have it checked (crop, planting date, NDVI picture, stress).",
        "description": ("Editable view of FieldChecks. Draw a field, choose the crop and, if known, the planting "
                        "date; the notebook 'RW Crop Field Check' fills in the results within about ten minutes."),
        "tags": TAGS})
    return view


def set_field_defaults(url, defaults):
    """Give fields a default value, so a form only asks for what the user really has to type."""
    if DRY_RUN or not url:
        return
    info = layer_info(url)
    fields, changed = [], []
    for f in info.get("fields", []):
        want = defaults.get(f["name"])
        if want is not None and f.get("defaultValue") != want:
            fields.append(dict(f, defaultValue=want))
            changed.append(f["name"])
    if not fields:
        return
    admin = url.replace("/rest/services/", "/rest/admin/services/")
    try:
        rest(admin + "/updateDefinition", {"updateDefinition": {"fields": fields}})
        _INFO.pop(url, None)
        log(f"  default values set on {', '.join(changed)}")
    except Exception as exc:
        log(f"  could not set default values ({exc})")


def set_edit_template(url, name, description, prototype=None, tool="esriFeatureEditToolPolygon"):
    """Give a layer an editing template. Without one, Map Viewer and Field Maps do not offer it for drawing."""
    if DRY_RUN or not url:
        return
    info = layer_info(url)
    want = [{"name": name, "description": description, "drawingTool": tool,
             "prototype": {"attributes": prototype or {}}}]
    if info.get("templates") == want:
        return
    admin = url.replace("/rest/services/", "/rest/admin/services/")
    try:
        rest(admin + "/updateDefinition", {"updateDefinition": {"templates": want}})
        _INFO.pop(url, None)
        log(f"  drawing template '{name}' set on {url.rsplit('/', 2)[-2]}/{url.rsplit('/', 1)[-1]}")
    except Exception as exc:
        log(f"  could not set the drawing template ({exc})")


def sync_view(view, item):
    """A view keeps the layers and fields it was created with: add the service's newer layers and show its newer fields."""
    admin = view.url.rstrip("/").replace("/rest/services/", "/rest/admin/services/")
    source = {l.properties.name: l for l in list(item.layers) + list(item.tables)}
    have = {l.properties.name: l for l in list(view.layers) + list(view.tables)}
    missing = [n for n in source if n not in have]
    if missing:
        log(f"  adding {missing} to the view")
        defs = [{"name": n, "id": source[n].properties.id, "adminLayerInfo": {"viewLayerDefinition": {
            "sourceServiceName": SERVICE_NAME, "sourceLayerId": source[n].properties.id, "sourceLayerFields": "*"}}}
            for n in missing]
        rest(admin + "/addToDefinition", {"addToDefinition": {"layers": [d for d, n in zip(defs, missing)
                                                                         if source[n].properties.type != "Table"],
                                                              "tables": [d for d, n in zip(defs, missing)
                                                                         if source[n].properties.type == "Table"]}})
    sync_view_fields(view, item)


def sync_view_fields(view, item, layers=None):
    """Show the fields the service gained after the view was created (all its layers, or only `layers`)."""
    if view is None or item is None or DRY_RUN:
        return
    admin = view.url.rstrip("/").replace("/rest/services/", "/rest/admin/services/")
    source = {l.properties.name: l for l in list(item.layers) + list(item.tables)}
    for lyr in list(view.layers) + list(view.tables):
        n = lyr.properties.name
        if n not in source or (layers is not None and n not in layers):
            continue
        shown = {f["name"].lower() for f in lyr.properties.fields}
        new = [f["name"] for f in source[n].properties.fields if f["name"].lower() not in shown]
        if new:
            log(f"  showing {new} in {view.title}'s {n}")
            rest(f"{admin}/{lyr.properties.id}/updateDefinition",
                 {"updateDefinition": {"fields": [{"name": f, "visible": True} for f in new]}})


ADMIN = {}
ADMIN_LAYER_IDS = {"cell": 0, "district": 1, "province": 2, "sector": 3, "village": 4}
PROVINCE_NAMES = {"1": "City of Kigali", "2": "Southern Province", "3": "Western Province",
                  "4": "Northern Province", "5": "Eastern Province"}


def admin_init():
    item = gis.content.get(ADMIN_ITEM_ID)
    if item is None:
        raise RuntimeError(f"Admin boundaries item {ADMIN_ITEM_ID} not found")
    ADMIN["url"] = item.url.rstrip("/")
    ADMIN["item"] = item


def admin_url(level):
    return f"{ADMIN['url']}/{ADMIN_LAYER_IDS[level]}"


def admin_attrs(row):
    a = lower_keys(row)
    pid = str(a.get("province_id") or "")
    out = {"province": PROVINCE_NAMES.get(pid, a.get("province")), "province_id": pid,
           "district": a.get("district"), "district_id": a.get("district_id"),
           "sector": a.get("sector"), "sector_id": a.get("sector_id"),
           "cell": a.get("cell"), "cell_id": a.get("cell_id"),
           "village": a.get("village"), "village_id": a.get("village_id")}
    if out["sector"]:
        out["sector_label"] = f"{out['sector']} ({out['district']})"
    if out["village"]:
        out["village_label"] = f"{out['village']} ({out['cell']})"
    return out


def village_at(x, y):
    """Admin attributes of the village containing a WGS84 point (None if outside Rwanda's villages)."""
    res = rest(admin_url("village") + "/query", {
        "geometry": {"x": x, "y": y, "spatialReference": {"wkid": 4326}}, "geometryType": "esriGeometryPoint",
        "inSR": 4326, "spatialRel": "esriSpatialRelIntersects", "outFields": "*", "returnGeometry": "false"})
    feats = res.get("features", [])
    return admin_attrs(feats[0]["attributes"]) if feats else None


def villages_near(geom4326, distance_m=0):
    res = rest(admin_url("village") + "/query", {
        "geometry": dict(geom4326, spatialReference={"wkid": 4326}), "geometryType": "esriGeometryPolygon",
        "inSR": 4326, "spatialRel": "esriSpatialRelIntersects", "distance": distance_m or None,
        "units": "esriSRUnit_Meter" if distance_m else None, "outFields": "*", "returnGeometry": "false"})
    return [admin_attrs(f["attributes"]) for f in res.get("features", [])]


RUN_MEANINGS = {
    "success": "The run finished and wrote the values of new Sentinel-2 images",
    "no_new_data": "The run finished; there was no new image with at most 20% cloud to process",
    "partial": "The run finished but some rows could not be saved; see the message",
    "failed": "The run stopped with an error; see the message. The same images are tried again next run",
    "skipped": "Nothing to do this time (for example, no season is due for classification)",
    "running": "The run is still in progress; values are saved after every image date",
}
RUN_LOCK = {}
LOCK_STALE_MIN = 90


def run_lock(parts, stage, scope="service"):
    """One run at a time: a RunLog row with status 'running' and a recent heartbeat blocks new runs, so scheduled
    back-fills can start every few minutes without overlapping."""
    url = parts.get("RunLog")
    if not url or DRY_RUN:
        return True
    since = dt.datetime.now(UTC) - dt.timedelta(minutes=LOCK_STALE_MIN)
    where = f"status = 'running' AND finished_utc >= {ts_sql(since)}"
    if scope == "stage":
        where += f" AND stage = '{stage}'"
    busy = query_all(url, where, "run_id,stage,started_utc")
    if busy:
        a = lower_keys(busy[0]["attributes"])
        log(f"Another run ({a['run_id']}, {a['stage']}) is still in progress; this run stops here.")
        return False
    now = to_ms(dt.datetime.now(UTC))
    res = rest(url + "/applyEdits", {"adds": [{"attributes": {
        "run_id": RUN_ID, "stage": stage, "status": "running", "run_mode": RUN_MODE, "started_utc": to_ms(RUN_START),
        "meaning": RUN_MEANINGS["running"] + ".",
        "finished_utc": now, "message": f"Run in progress ({NOTEBOOK_TITLE})"}}]})
    RUN_LOCK.update(url=url, oid=res["addResults"][0].get("objectId"), stage=stage)
    return True


def heartbeat(summary=""):
    """Keep the run lock alive and show progress in RunLog (the dashboards' last-run indicator)."""
    if not RUN_LOCK.get("oid") or DRY_RUN:
        return
    try:
        rest(RUN_LOCK["url"] + "/applyEdits", {"updates": [{"attributes": {
            oid_field(RUN_LOCK["url"]): RUN_LOCK["oid"], "finished_utc": to_ms(dt.datetime.now(UTC)),
            "message": (f"Run in progress: {summary}\n---\n" + "\n".join(_LOG[-20:]))[-3990:]}}]})
    except Exception as exc:
        log(f"  heartbeat failed: {exc}")


def runlog_write(parts, stage, status, summary="", **counts):
    now = dt.datetime.now(UTC)
    if EDIT_ERRORS and status in ("success", "no_new_data"):
        status = "partial"
        summary += f" | {len(EDIT_ERRORS)} rows could not be written, first error: {EDIT_ERRORS[0]}"
    tail = "\n".join(_LOG[-40:])
    row = {"run_id": RUN_ID, "stage": stage, "status": status, "run_mode": RUN_MODE,
           "meaning": RUN_MEANINGS.get(status, status) + f" ({counts.get('dates', 0)} image dates, "
                      f"{counts.get('observations', 0)} rows written)",
           "started_utc": to_ms(RUN_START), "finished_utc": to_ms(now),
           "dates": counts.get("dates", 0), "scenes": counts.get("scenes", 0), "units": counts.get("units", 0),
           "observations": counts.get("observations", 0), "alerts": counts.get("alerts", 0),
           "latest_image": to_ms(counts.get("latest_image")),
           "message": (summary + "\n---\n" + tail)[-3990:]}
    log(f"[{stage}] {status}: {summary}")
    if not parts.get("RunLog"):
        return
    if RUN_LOCK.get("oid"):
        row["OBJECTID"] = RUN_LOCK.pop("oid")
        apply_edits(parts["RunLog"], updates=[{"attributes": row}])
    else:
        apply_edits(parts["RunLog"], adds=[{"attributes": row}])


def fill_meanings(url, fn, limit=50000):
    """Fill `meaning` on rows written before the field existed (or where it is empty)."""
    if not url or DRY_RUN:
        return
    rows = query_all(url, "meaning IS NULL")[:limit]
    if not rows:
        return
    oid = oid_field(url)
    updates = []
    for r in rows:
        a = lower_keys(r["attributes"])
        text = fn(a)
        if text:
            updates.append({"attributes": {oid: a[oid.lower()], "meaning": text[:800]}})
    apply_edits(url, updates=updates, batch=1000)
    log(f"  explained {len(updates)} existing rows in {url.rsplit('/', 1)[-1]}")


def fill_common_meanings(parts):
    fill_meanings(parts.get("SceneLog"), scene_meaning)
    fill_meanings(parts.get("RunLog"), lambda a: RUN_MEANINGS.get(a.get("status"), a.get("status")) + ".")
    fill_meanings(parts.get("RasterRegistry"), registry_meaning)


def guide_sync(parts):
    url = parts.get("IndexGuide")
    if not url or DRY_RUN:
        return
    lengths = {f["name"].lower(): f.get("length") for f in layer_info(url).get("fields", []) if f.get("length")}
    want = [{k: v[:lengths[k]] if isinstance(v, str) and lengths.get(k) else v for k, v in dict(g, sort_order=i).items()}
            for i, g in enumerate(INDEX_GUIDE, start=1)]
    keys = sorted({k for g in want for k in g})
    rows = query_all(url, fields=",".join(keys))

    def content(entries):
        return sorted(tuple("" if e.get(k) is None else str(e.get(k)) for k in keys) for e in entries)
    if content(lower_keys(r["attributes"]) for r in rows) == content(want):
        return
    log(f"  refreshing IndexGuide ({len(INDEX_GUIDE)} entries)")
    apply_edits(url, deletes=[lower_keys(r["attributes"])["objectid"] for r in rows])
    apply_edits(url, adds=[{"attributes": dict(g, sort_order=i)} for i, g in enumerate(INDEX_GUIDE, start=1)])


def registry_add(parts, kind, item, label="", season="", day="", cell_m=None, note="", exclusive=False):
    url = parts.get("RasterRegistry")
    if not url or item is None:
        return
    if exclusive:
        old = query_all(url, f"kind = '{kind}' AND active = 1" + (f" AND season = '{season}'" if season else ""))
        apply_edits(url, updates=[{"attributes": {"OBJECTID": lower_keys(r["attributes"])["objectid"], "active": 0}}
                                  for r in old])
    row = {"kind": kind, "label": label, "season": season, "acq_day": day, "item_id": item.id,
           "url": (item.url or "")[:400], "title": item.title, "cell_m": cell_m, "active": 1,
           "created_utc": to_ms(dt.datetime.now(UTC)), "note": note[:400]}
    apply_edits(url, adds=[{"attributes": dict(row, meaning=registry_meaning(row))}])


REGISTRY_KINDS = {
    "zone_raster": "raster of unit codes used to compute statistics for every unit in one job",
    "village_raster": "raster of village codes used to build crop zone codes",
    "class_raster": "crop type map produced by the classification model",
    "ndvi_overlay": "processed NDVI image over the monitored units, shown on the web map",
    "water_overlay": "each water body on its latest clear image (band 1 red minus SWIR reflectance for turbidity, "
                     "band 2 NDCI for chlorophyll, clean water pixels only; bands 3 and 4 are the same two "
                     "indices rescaled to that water body's own spread on that date), shown on the web map",
    "change_raster": "difference of each index between the current and the baseline period",
    "evidence_raster": "mining-likelihood score of each factor and the combined risk index",
    "risk_class": "risk class of every pixel: 1 Low, 2 Medium, 3 High, 0 excluded (settlement, road, stable cropland, water)",
    "slope": "terrain slope in degrees from the Copernicus 30 m DEM",
    "dist_river": "distance in metres to the nearest mapped river",
    "dist_road": "distance in metres to the nearest main road",
}


def registry_meaning(r):
    what = REGISTRY_KINDS.get(r.get("kind"), "raster created by this notebook")
    when = f" for {r['season'] or r['acq_day']}" if (r.get("season") or r.get("acq_day")) else ""
    state = "in use" if r.get("active") else "replaced by a newer one"
    return f"{r.get('title')}: {what}{when}, {r.get('cell_m') or '?'} m cells, {state}."


def registry_get(parts, kind, season=None, active=True):
    url = parts.get("RasterRegistry")
    if not url:
        return []
    where = f"kind = '{kind}'" + (" AND active = 1" if active else "") + (f" AND season = '{season}'" if season else "")
    rows = [lower_keys(r["attributes"]) for r in query_all(url, where)]
    return sorted(rows, key=lambda r: r.get("created_utc") or 0, reverse=True)

## 7. Web map and dashboard helpers
Your edits to the web map and dashboard are kept. Set `REBUILD_APPS = True` to regenerate both from scratch.

In [ ]:
def merc(lon, lat):
    return lon * 20037508.342789 / 180.0, math.log(math.tan((90.0 + lat) * math.pi / 360.0)) * 6378137.0


_x0, _y0 = merc(RWANDA_ENV["xmin"], RWANDA_ENV["ymin"])
_x1, _y1 = merc(RWANDA_ENV["xmax"], RWANDA_ENV["ymax"])
RWANDA_3857 = {"xmin": _x0, "ymin": _y0, "xmax": _x1, "ymax": _y1, "spatialReference": {"wkid": 102100, "latestWkid": 3857}}


def hexrgba(h, alpha=255):
    h = h.lstrip("#")
    return [int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16), alpha]


def sls(rgba, width=0.5):
    return {"type": "esriSLS", "style": "esriSLSSolid", "color": list(rgba), "width": width}


def sfs(rgba, outline=(60, 60, 60, 170), width=0.4):
    return {"type": "esriSFS", "style": "esriSFSSolid", "color": list(rgba), "outline": sls(outline, width)}


def hollow(rgba, width):
    return {"type": "esriSFS", "style": "esriSFSNull", "color": [0, 0, 0, 0], "outline": sls(rgba, width)}


def unique_renderer(field, classes, alpha=190, default_label="Not yet observed", outline_width=None):
    """classes: [(value, '#hex')]."""
    def symbol(c):
        if outline_width:
            return sfs(hexrgba(c, alpha), hexrgba(c, 255), outline_width)
        return sfs(hexrgba(c, alpha))
    return {"type": "uniqueValue", "field1": field,
            "defaultSymbol": sfs((189, 189, 189, 60), (120, 120, 120, 200), outline_width or 0.4),
            "defaultLabel": default_label,
            "uniqueValueInfos": [{"value": v, "label": v, "symbol": symbol(c)} for v, c in classes]}


def popup(title, rows, note_field=None):
    """rows: [(field, label, digits | 'date' | None)]"""
    infos = []
    for field, label, fmt in rows:
        info = {"fieldName": field, "label": label, "visible": True}
        if fmt == "date":
            info["format"] = {"dateFormat": "shortDate"}
        elif isinstance(fmt, int):
            info["format"] = {"places": fmt, "digitSeparator": True}
        infos.append(info)
    elements = []
    if note_field:
        elements.append({"type": "text", "text": "<p>{" + note_field + "}</p>"})
    elements.append({"type": "fields"})
    return {"title": title, "fieldInfos": infos, "popupElements": elements, "showAttachments": False}


def op_feature(lid, title, url, item_id, renderer=None, popup_info=None, visible=True, opacity=0.9,
               min_scale=0, max_scale=0):
    lyr = {"id": lid, "title": title, "url": url, "layerType": "ArcGISFeatureLayer", "visibility": visible,
           "opacity": opacity, "minScale": min_scale, "maxScale": max_scale,
           "layerDefinition": {"minScale": min_scale, "maxScale": max_scale}}
    if item_id:
        lyr["itemId"] = item_id
    if renderer:
        lyr["layerDefinition"]["drawingInfo"] = {"renderer": renderer}
    if popup_info:
        lyr["popupInfo"] = popup_info
    else:
        lyr["disablePopup"] = True
    return lyr


def op_imagery(lid, title, url, item_id, rule, mosaic=None, visible=False, opacity=1.0, min_scale=None):
    lyr = {"id": lid, "title": title, "url": url, "layerType": "ArcGISImageServiceLayer", "visibility": visible,
           "opacity": opacity, "format": "png32", "compressionQuality": 75, "renderingRule": rule,
           "timeAnimation": False}
    if min_scale:
        lyr.update(minScale=min_scale, maxScale=0)
    if item_id:
        lyr["itemId"] = item_id
    if mosaic:
        lyr["mosaicRule"] = mosaic
    return lyr


def op_table(tid, title, url, item_id):
    return {"id": tid, "title": title, "url": url, "itemId": item_id, "layerDefinition": {}}


def s2_live_layers():
    """Always-current Sentinel-2 context layers straight from the image service (no storage used)."""
    url, iid = S2["url"], S2_ITEM_ID
    mosaic = s2_live_mosaic()
    return [op_imagery("s2_true", "Sentinel-2 natural colour (latest clear)", url, iid,
                       {"rasterFunction": "Natural Color for Visualization"}, mosaic, visible=False),
            op_imagery("s2_ndvi", "Sentinel-2 NDVI (latest clear)", url, iid,
                       {"rasterFunction": "NDVI Colorized for Visualization"}, mosaic, visible=False, opacity=0.85)]


def admin_layers():
    base, iid = ADMIN["url"], ADMIN_ITEM_ID
    return [
        op_feature("admin_district", "District boundaries", f"{base}/1", iid,
                   {"type": "simple", "symbol": hollow((33, 33, 33, 230), 1.3)}, opacity=1),
        op_feature("admin_sector", "Sector boundaries", f"{base}/3", iid,
                   {"type": "simple", "symbol": hollow((66, 66, 66, 210), 0.8)}, opacity=1, min_scale=700000,
                   popup_info=popup("Sector {sector}", [("district", "District", None), ("sector", "Sector", None)])),
        op_feature("admin_village", "Village boundaries", f"{base}/4", iid,
                   {"type": "simple", "symbol": hollow((110, 110, 110, 190), 0.4)}, opacity=1, min_scale=90000,
                   popup_info=popup("Village {village}", [("sector", "Sector", None), ("cell", "Cell", None),
                                                          ("village", "Village", None)])),
    ]


def backup_json(item):
    """Save an item's current JSON to <home>/backups/<itemId>_<timestamp>.json before it is overwritten."""
    home = "/arcgis/home" if os.path.isdir("/arcgis/home") else os.getcwd()
    folder = os.path.join(home, "backups")
    os.makedirs(folder, exist_ok=True)
    data = item.get_data()
    data = json.loads(data) if isinstance(data, str) else data
    path = os.path.join(folder, f"{item.id}_{dt.datetime.now(UTC):%Y%m%dT%H%M%SZ}.json")
    with open(path, "w") as fh:
        json.dump(data, fh, indent=1)
    log(f"  backup of '{item.title}' saved to {path}")
    return path


def add_item(title, item_type, data, snippet, type_keywords):
    text = json.dumps(data)
    try:
        from arcgis.gis import ItemProperties
        folder = gis.content.folders.get(FOLDER) or gis.content.folders.create(FOLDER, exist_ok=True)
        props = ItemProperties(title=title, item_type=item_type, tags=TAGS.split(","), snippet=snippet,
                               type_keywords=type_keywords, extent=[[RWANDA_ENV["xmin"], RWANDA_ENV["ymin"]],
                                                                    [RWANDA_ENV["xmax"], RWANDA_ENV["ymax"]]], text=text)
        return folder.add(props).result()
    except Exception as exc:
        log(f"  creating {title} with the classic API ({str(exc)[:90]})")
        return gis.content.add(item_properties={"title": title, "type": item_type, "tags": TAGS, "snippet": snippet,
                                                "typeKeywords": ",".join(type_keywords), "text": text}, folder=FOLDER)


def ensure_webmap(layers, tables):
    item = find_item(WEBMAP_TITLE, "Web Map")
    if DRY_RUN:
        return item
    basemap = dict(gis.properties.get("defaultBasemap") or {})
    basemap = basemap or {"title": "Topographic", "baseMapLayers": [{
        "id": "topo", "layerType": "ArcGISTiledMapServiceLayer", "title": "World Topographic Map", "visibility": True,
        "url": "https://services.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer"}]}
    data = {"operationalLayers": layers, "tables": tables, "baseMap": basemap,
            "spatialReference": {"wkid": 102100, "latestWkid": 3857}, "version": "2.27",
            "authoringApp": "GeoHub monitoring notebook", "authoringAppVersion": "1.0",
            "initialState": {"viewpoint": {"targetGeometry": RWANDA_3857}}}
    if item is None:
        log(f"Creating web map {WEBMAP_TITLE}")
        item = add_item(WEBMAP_TITLE, "Web Map", data, SERVICE_SNIPPET,
                        ["ArcGIS Online", "Explorer Web Map", "Map", "Online Map", "Web Map"])
    elif globals().get("REBUILD_WEBMAPS", REBUILD_APPS):
        log(f"Rebuilding web map {WEBMAP_TITLE}")
        backup_json(item)
        item.update(item_properties={"text": json.dumps(data)})
    return item


_USER_LAYER_KEYS = ("visibility", "opacity", "popupInfo", "disablePopup", "showLegend", "listMode",
                    "layerDefinition", "blendMode", "effect", "minScale", "maxScale", "refreshInterval")


def patch_webmap(item, replacements):
    """Replace operational layers by id in an existing web map (adds them if missing)."""
    if item is None or DRY_RUN or not replacements:
        return
    data = item.get_data()
    if isinstance(data, str):
        data = json.loads(data)
    ops = data.get("operationalLayers", [])
    by_id = {lyr["id"]: lyr for lyr in replacements}
    out, seen = [], set()
    for lyr in ops:
        lid = lyr.get("id")
        if lid not in by_id:
            out.append(lyr)
            continue
        if lid in seen:
            continue
        fresh = by_id[lid]
        keep = {k: lyr[k] for k in _USER_LAYER_KEYS if k in lyr}
        if lyr.get("title") and lyr["title"].split(" (")[0] != fresh.get("title", "").split(" (")[0]:
            keep["title"] = lyr["title"]
        out.append(dict(fresh, **keep))
        seen.add(lid)
    for lyr in replacements:
        if lyr["id"] not in seen:
            out.insert(0, lyr)
    data["operationalLayers"] = out
    item.update(item_properties={"text": json.dumps(data)})


DASH_OID = "objectid"
_STATE = {"verticalAlignment": "middle", "showCaption": True, "showDescription": True}


def _uid():
    return str(uuid.uuid4())


def vfmt(digits=0, prefix=False):
    return {"name": "value", "prefix": prefix, "style": "decimal", "useGrouping": True, "maximumFractionDigits": digits}


def dash_filter(filters):
    """{field: value or [values]} on string fields -> Dashboards filter JSON (AND of ORs)."""
    groups = []
    for field, values in filters.items():
        if values == "__notnull__":
            groups.append({"type": "filterGroup", "condition": "OR", "rules": [
                {"type": "filterRule", "field": {"name": field, "type": "number"}, "operator": "is_not_null",
                 "constraint": {"type": "nil"}}]})
            continue
        values = values if isinstance(values, (list, tuple)) else [values]
        groups.append({"type": "filterGroup", "condition": "OR", "rules": [
            {"type": "filterRule", "field": {"name": field, "type": "string"}, "operator": "equal",
             "constraint": {"type": "value", "value": v}} for v in values]})
    return {"type": "filterGroup", "condition": "AND", "rules": groups}


def dset(src, group_by=None, order_by=None, stats=None, filters=None, max_features=None):
    d = {"type": "serviceDataset", "name": "main",
         "dataSource": {"type": "layerDataSource", "itemId": src[0], "layerId": src[1]},
         "groupByFields": group_by or [], "orderByFields": order_by or [], "statisticDefinitions": stats or [],
         "clientSideStatistics": False, "outFields": ["*"], "returnDistinctValues": False,
         "allowSourceDownload": True, "allowSummaryDownload": True}
    if max_features:
        d["maxFeatures"] = max_features
    if filters:
        d["filter"] = dash_filter(filters)
    return d


def widget(wtype, name, src=None, caption=None):
    w = {"id": _uid(), "name": name, "showLastUpdate": wtype not in ("categorySelectorWidget", "mapWidget"),
         "noDataState": dict(_STATE), "noFilterState": dict(_STATE), "type": wtype, "_src": src}
    if caption:
        w["caption"] = f'<p style="text-align:center;"><strong>{caption}</strong></p>'
    return w


def w_selector(label, src, field):
    w = widget("categorySelectorWidget", f"{label} selector", src)
    w.update({"caption": label, "showActionButtons": True, "events": [],
              "datasets": [dset(src, group_by=[field], order_by=[f"{field} ASC"], max_features=5000,
                                stats=[{"onStatisticField": field, "outStatisticFieldName": "count_result",
                                        "statisticType": "count"}])],
              "category": {"type": "groupByValues", "labelOverrides": []},
              "selection": {"operator": "equal", "allowNone": True, "noneLabel": "All", "noneLabelPlacement": "first",
                            "type": "single", "defaultSelection": "first"},
              "showFilter": True, "displayType": "list", "maxSize": "compact", "presentationMode": "dropdown",
              "_field": field})
    return w


def w_indicator(top, src, stat="count", field=None, filters=None, digits=0, bottom=""):
    field = field or DASH_OID
    w = widget("indicatorWidget", top, src)
    w.update({"datasets": [dset(src, filters=filters, stats=[{"onStatisticField": field,
                                                              "outStatisticFieldName": "value", "statisticType": stat}])],
              "defaultSettings": {"topSection": {"fontSize": 80, "textInfo": {"text": top}},
                                  "middleSection": {"fontSize": 160, "textInfo": {"text": "{calculated/value}"}},
                                  "bottomSection": {"fontSize": 80, "textInfo": {"text": bottom}}},
              "comparison": "none", "valueFormat": vfmt(digits),
              "percentageFormat": {"name": "percentage", "prefix": False, "style": "percent", "useGrouping": True},
              "ratioFormat": {"name": "ratio", "prefix": False, "style": "decimal", "useGrouping": True,
                              "maximumFractionDigits": 2},
              "valueType": "statistic", "noValueState": dict(_STATE)})
    return w


RUN_ARCADE = """var s = $datapoint.status;
var c = When(s == 'success', '#2e7d32', s == 'no_new_data', '#1565c0', s == 'running', '#6a1b9a', s == 'partial', '#ef6c00', '#c62828');
var label = Decode(s, 'success', 'Updated', 'no_new_data', 'No new clear images', 'running', 'Running now', 'partial', 'Partly updated', 'failed', 'Failed', s);
return {
  topText: 'Last automated run',
  middleText: label,
  middleTextColor: c,
  bottomText: $datapoint.stage + ' - ' + Text($datapoint.finished_utc, 'D MMM Y HH:mm'),
  bottomTextColor: '#666666'
}"""


def w_runlog(src):
    return w_latest("Last run", src, "finished_utc", RUN_ARCADE)


def w_latest(name, src, order_field, expression):
    """Indicator showing the newest row of a table through an Arcade expression."""
    w = widget("indicatorWidget", name, src)
    w.update({"datasets": [dset(src, order_by=[f"{order_field} DESC"], max_features=1)],
              "arcadeEnabled": True, "expression": expression, "arcadeRefreshEnabled": False,
              "defaultSettings": {"topSection": {"fontSize": 80, "textInfo": {}},
                                  "middleSection": {"fontSize": 160, "textInfo": {"text": "{calculated/value}"}},
                                  "bottomSection": {"fontSize": 80, "textInfo": {}}},
              "comparison": "none", "valueField": DASH_OID, "valueFormat": vfmt(0),
              "percentageFormat": {"name": "percentage", "prefix": False, "style": "percent", "useGrouping": True},
              "ratioFormat": {"name": "ratio", "prefix": False, "style": "decimal", "useGrouping": True},
              "valueType": "feature", "noValueState": dict(_STATE), "iconInfos": []})
    return w


_DATE_PATTERNS = [{"period": "ss", "pattern": "HH:mm:ss"}, {"period": "mm", "pattern": "HH:mm"},
                  {"period": "hh", "pattern": "HH:mm"}, {"period": "DD", "pattern": "d MMM"},
                  {"period": "MM", "pattern": "MMM yyyy"}, {"period": "YYYY", "pattern": "yyyy"}]


def _serial(w, src, group_by, order_by, stats, filters, category, parse_dates, y_title, digits, split,
            graph_type, color, rotate=False, max_features=None):
    w.update({
        "datasets": [dset(src, group_by=group_by, order_by=order_by, stats=stats, filters=filters,
                          max_features=max_features)],
        "selectionMode": "single", "categoryType": "groupByValues",
        "category": dict({"fieldName": category, "nullLabel": "null", "blankLabel": "blank", "defaultColor": "#d6d6d6",
                          "nullColor": "#d6d6d6", "blankColor": "#d6d6d6", "labelOverrides": [],
                          "byCategoryColors": False, "labelsPlacement": "default", "labelRotation": 0}),
        "valueFormat": vfmt(digits), "labelFormat": {"name": "label", "prefix": True, "style": "decimal",
                                                     "useGrouping": True, "maximumFractionDigits": digits},
        "datePeriodPatterns": _DATE_PATTERNS,
        "chartScrollbar": {"enabled": True, "dragIcon": "dragIconRoundSmall", "dragIconHeight": 20,
                           "dragIconWidth": 20, "scrollbarHeight": 15},
        "categoryAxis": {"titleRotation": 0, "gridPosition": "start", "gridThickness": 1, "gridAlpha": 0.15,
                         "axisThickness": 1, "axisAlpha": 0.5, "labelsEnabled": True, "parseDates": parse_dates,
                         "minPeriod": "DD"},
        "valueAxis": {"title": y_title, "titleRotation": 270, "gridThickness": 1, "gridAlpha": 0.15,
                      "axisThickness": 1, "axisAlpha": 0.5, "labelsEnabled": True, "stackType": "none",
                      "integersOnly": False, "logarithmic": False},
        "legend": {"enabled": bool(split), "position": "bottom", "markerSize": 15, "markerType": "circle",
                   "align": "center", "labelWidth": 100, "valueWidth": 0},
        "seriesOrderByFields": [], "guides": [], "rotate": rotate,
        "graphs": [] if split else [{"lineColorField": "_lineColor_", "fillColorsField": "_fillColor_",
                                     "type": graph_type, "fillAlphas": 1 if graph_type == "column" else 0,
                                     "lineAlpha": 1, "lineThickness": 2, "bullet": "round" if graph_type == "line" else "none",
                                     "bulletAlpha": 1, "bulletBorderAlpha": 0, "bulletBorderThickness": 2,
                                     "showBalloon": True, "bulletSize": 6, "connect": True, "valueField": "value",
                                     "lineColor": color, "fillColors": color}],
        "splitBy": ({"fieldName": split[0], "defaultColor": "#d6d6d6",
                     "seriesProperties": [{"key": k, "label": k, "color": c} for k, c in split[1]]}
                    if split else {"defaultColor": "#d6d6d6", "seriesProperties": []}),
        "commonGraphProperties": {"lineColorField": "_lineColor_", "fillColorsField": "_fillColor_",
                                  "type": graph_type, "fillAlphas": 1 if graph_type == "column" else 0,
                                  "lineAlpha": 1, "lineThickness": 2,
                                  "bullet": "round" if graph_type == "line" else "none", "bulletAlpha": 1,
                                  "bulletBorderAlpha": 0, "bulletBorderThickness": 2, "showBalloon": True,
                                  "bulletSize": 6, "connect": True}})
    return w


def w_timeseries(caption, src, date_field, value_field, stat="avg", filters=None, y_title="", digits=3,
                 split=None, color="#2e7d32"):
    w = widget("serialChartWidget", caption, src, caption)
    group = [date_field] + ([split[0]] if split else [])
    stats = [{"onStatisticField": value_field, "outStatisticFieldName": "value", "statisticType": stat}]
    return _serial(w, src, group, [f"{date_field} ASC"], stats, filters, date_field, True, y_title, digits,
                   split, "line", color)


def w_bars(caption, src, category, stat="count", value_field=None, filters=None, y_title="", digits=0,
           color="#546e7a", colors=None):
    value_field = value_field or DASH_OID
    w = widget("serialChartWidget", caption, src, caption)
    stats = [{"onStatisticField": value_field, "outStatisticFieldName": "value", "statisticType": stat}]
    _serial(w, src, [category], ["value DESC"], stats, filters, category, False, y_title, digits, None,
            "column", color)
    if colors:
        w["category"]["byCategoryColors"] = True
        w["category"]["labelOverrides"] = [{"key": k, "label": k, "color": c} for k, c in colors]
    return w


def w_rank(caption, src, category, value_field, stat="max", filters=None, y_title="", digits=1, top=None,
           ascending=False, color="#546e7a"):
    """Horizontal bars, one per category, sorted by value (highest first unless ascending)."""
    w = widget("serialChartWidget", caption, src, caption)
    stats = [{"onStatisticField": value_field, "outStatisticFieldName": "value", "statisticType": stat}]
    _serial(w, src, [category], ["value ASC" if ascending else "value DESC"], stats,
            dict(filters or {}, **{value_field: "__notnull__"}), category, False, y_title, digits, None, "column",
            color, rotate=True, max_features=top)
    return w


def palette(keys):
    """Distinct colours for a split-by chart: [(key, colour)]."""
    import colorsys
    keys = list(keys)
    out = []
    for i, k in enumerate(keys):
        r, g, b = colorsys.hsv_to_rgb((i * 0.61803) % 1.0, 0.75, 0.55 + 0.35 * ((i * 7) % 3) / 2)
        out.append((k, "#%02x%02x%02x" % (int(r * 255), int(g * 255), int(b * 255))))
    return out


def _column(field, title, digits=None):
    c = {"title": title, "fieldName": field, "textAlign": "left" if digits is None else "right",
         "isBold": False, "isItalic": False, "isUnderlined": False}
    if digits is not None:
        c["valueFormat"] = vfmt(digits)
    return c


_TABLE_SETTINGS = {"layout": "fit-columns", "scale": "medium", "hoverText": False, "rowStriping": True,
                   "selectionMode": "single", "verticalGrid": {"alpha": 1, "thickness": 1},
                   "horizontalGrid": {"alpha": 1, "thickness": 1}, "headerSettings": {"rule": {"alpha": 1, "thickness": 3}}}


def w_table(caption, src, columns, filters=None, order_by=None, max_features=2000):
    """columns: [(field, title, digits|None)]"""
    w = widget("tableWidget", caption, src, caption)
    w.update({"datasets": [dset(src, order_by=order_by, filters=filters, max_features=max_features)],
              "dataSettings": {"type": "features", "valueFields": [c[0] for c in columns]},
              "tableSettings": dict(_TABLE_SETTINGS), "columnSettings": [_column(*c) for c in columns]})
    return w


def w_group_table(caption, src, group_field, group_title, columns, filters=None):
    """One row per distinct group_field value. columns: [(field, title, statistic, digits)]"""
    w = widget("tableWidget", caption, src, caption)
    stats = [{"onStatisticField": f, "outStatisticFieldName": f"{s}_{f}", "statisticType": s} for f, _, s, _ in columns]
    cols = [_column(group_field, group_title)]
    for f, title, s, digits in columns:
        c = _column(f, title, digits)
        c["statisticsKey"] = f"{s}_{f}"
        cols.append(c)
    w.update({"datasets": [dset(src, group_by=[group_field], order_by=[f"{group_field} ASC"], stats=stats,
                                filters=filters)],
              "dataSettings": {"type": "groupByValues", "fieldName": group_field, "rule": {"alpha": 1, "thickness": 3},
                               "labelOverrides": []},
              "tableSettings": dict(_TABLE_SETTINGS, summarySettings={"label": "Total", "rule": {"alpha": 1, "thickness": 3}}),
              "columnSettings": cols})
    return w


def w_map(webmap_id):
    w = widget("mapWidget", "Map")
    w.update({"flashRepeats": 3, "itemId": webmap_id,
              "mapTools": [{"type": "legendTool"}, {"type": "mapContentsTool"}, {"type": "searchTool"}],
              "showNavigation": True, "showPanRotate": False, "showLocate": False, "showCompass": False,
              "showPopup": True, "scalebarStyle": "line",
              "selectTool": {"type": "selectTool", "clickToSelect": True, "advanceSketchTools": []}})
    return w


def connect_filter(source, targets, zoom_to=None):
    """targets: [(widget, field_map or None, map_layer_id or None)]"""
    out = []
    for tw, field_map, map_layer in targets:
        t = {"targetId": f"{tw['id']}#{map_layer or 'main'}", "by": "whereClause", "requiresSelection": False}
        if field_map and (map_layer or tw.get("_src") != source.get("_src")):
            t["fieldMap"] = [{"sourceName": s, "targetName": d} for s, d in field_map]
        out.append(t)
    actions = [{"type": "filter", "targets": out}] if out else []
    if zoom_to is not None:
        actions += [{"type": t, "targets": [{"targetId": zoom_to["id"], "by": "geometry",
                                             "requiresSelection": True}]}
                    for t in ("zoom", "flashGeometry")]
    if actions:
        source.setdefault("events", []).append({"type": "selectionChanged", "actions": actions})


def lay_item(w, width=1.0, height=1.0, tab=None):
    d = {"type": "itemLayoutElement", "id": w["id"], "width": width, "height": height}
    if tab:
        d["tabName"] = tab
    return d


def lay_stack(orientation, elements, width=1.0, height=1.0):
    return {"type": "stackLayoutElement", "orientation": orientation, "id": _uid(),
            "width": width, "height": height, "elements": elements}


def lay_tabs(elements, width=1.0, height=1.0):
    return {"type": "tabsLayoutElement", "id": _uid(), "width": width, "height": height, "elements": elements}


NUMBER_PREFIX = [{"key": k, "symbol": s, "enabled": e} for k, s, e in [
    ("yotta", "Y", True), ("zeta", "Z", True), ("exa", "E", True), ("peta", "P", True), ("tera", "T", True),
    ("giga", "G", True), ("mega", "M", True), ("kilo", "k", True), ("base", "", True), ("deci", "d", False),
    ("centi", "c", False), ("milli", "m", False), ("micro", "µ", False), ("nano", "n", False)]]


def build_dashboard(selectors, widgets, layout_root, theme_overrides=None, header_colors=None):
    """`theme_overrides` and `header_colors` give a project its own colours (see the forest dashboard)."""
    def clean(w):
        return {k: v for k, v in w.items() if not k.startswith("_")}
    header = {"type": "header", "title": DASHBOARD_TITLE, "subtitle": DASHBOARD_SUBTITLE,
              "logoSize": "small", "backgroundImageSizing": "fit-height",
              "normalBackgroundImagePlacement": "left", "horizontalBackgroundImagePlacement": "top",
              "showSignOutMenu": True, "selectors": [clean(s) for s in selectors]}
    header.update(header_colors or {})
    return {"version": 59, "authoringApp": "ArcGIS Dashboards", "authoringAppVersion": "4.29.0",
            "maxPaginationRecords": 50000, "timeZone": "system",
            "mapOverrides": {"highlightColor": "#ff00ff", "trackedFeatureColor": "#0000ff", "trackedFeatureRadius": 100},
            "theme": "light", "themeOverrides": theme_overrides or {}, "numberPrefixOverrides": NUMBER_PREFIX,
            "elementMappings": {},
            "desktopView": {"type": "desktopView", "widgets": [clean(w) for w in widgets],
                            "layout": {"type": "dockingLayout", "rootElement": layout_root},
                            "settings": {"allowElementResizing": False, "allowElementExpansion": True, "allowReset": True},
                            "header": header}}


def validate_dashboard(data):
    """Every widget must appear exactly once in the layout and every layout element must be a widget."""
    view = data["desktopView"]
    widget_ids = [w["id"] for w in view["widgets"]]
    selector_ids = [s["id"] for s in view["header"].get("selectors", [])]
    layout_ids = []

    def walk(el):
        if el.get("type") == "itemLayoutElement":
            layout_ids.append(el["id"])
        for child in el.get("elements", []):
            walk(child)

    walk(view["layout"]["rootElement"])
    problems = []
    if len(set(widget_ids + selector_ids)) != len(widget_ids + selector_ids):
        problems.append("duplicate element ids")
    if sorted(widget_ids) != sorted(layout_ids):
        problems.append(f"widgets not in layout: {set(widget_ids) - set(layout_ids)}; "
                        f"layout items without widget: {set(layout_ids) - set(widget_ids)}")
    known = set(widget_ids + selector_ids)
    for el in view["widgets"] + view["header"].get("selectors", []):
        for ev in el.get("events", []):
            for act in ev.get("actions", []):
                for tid in [t["targetId"] for t in act.get("targets", [])] + \
                           ([act["targetId"]] if act.get("targetId") else []):
                    if tid.split("#")[0] not in known:
                        problems.append(f"{el['name']} targets unknown element {tid}")
    if problems:
        raise ValueError("Dashboard JSON is invalid: " + "; ".join(problems))
    json.loads(json.dumps(data))
    return True


def cascade_selectors(selectors, targets_for):
    """Each selector filters the selectors after it and every target returned by targets_for(field)."""
    for i, sel in enumerate(selectors):
        field = sel["_field"]
        downstream = [(s, [(field, field)], None) for s in selectors[i + 1:]]
        connect_filter(sel, downstream + targets_for(field))


def ensure_dashboard(make):
    """make() -> dashboard JSON."""
    global DASH_OID
    target = globals().get("DASHBOARD_TARGET_ID")
    item = gis.content.get(target) if target else find_item(DASHBOARD_TITLE, "Dashboard")
    if DRY_RUN or (item is not None and not REBUILD_APPS):
        return item
    main_layer = SCHEMA["layers"][0]["name"]
    view = find_item(SERVICE_NAME + "_View", "Feature Service")
    if view is not None:
        DASH_OID = oid_field(f"{view.url.rstrip('/')}/{PARTS_ID[main_layer]}")
    data = make()
    validate_dashboard(data)
    if item is None:
        log(f"Creating dashboard {DASHBOARD_TITLE}")
        item = add_item(DASHBOARD_TITLE, "Dashboard", data, SERVICE_SNIPPET,
                        ["Dashboard", "Operations Dashboard", "ArcGIS Dashboards"])
    else:
        log(f"Rebuilding dashboard {DASHBOARD_TITLE}")
        backup_json(item)
        item.update(item_properties={"text": json.dumps(data)})
    return item


def ensure_task():
    """Create the daily notebook task once. TASK_CRON_UTC is "minute hour" in UTC (Kigali = UTC+2)."""
    if not CREATE_TASK or DRY_RUN:
        return
    nb = find_item(NOTEBOOK_TITLE, "Notebook")
    if nb is None:
        log(f"Task not created: notebook item '{NOTEBOOK_TITLE}' not found (save the notebook with that title)")
        return
    tasks = gis.users.me.tasks
    for t in tasks.all:
        if dict(t.properties).get("itemId") == nb.id:
            log(f"Scheduled task already exists for {NOTEBOOK_TITLE}")
            return
    minute, hour = TASK_CRON_UTC.split()[:2]
    res = gis._con.post(f"{tasks._url}/createTask", {
        "f": "json", "title": f"{NOTEBOOK_TITLE} ({TASK_LABEL})", "type": "ExecuteNotebook", "itemId": nb.id,
        "minute": minute, "hour": hour, "dayOfMonth": "*", "month": "*", "dayOfWeek": "?", "maxOccurrences": -1,
        "parameters": json.dumps({"updatePortalItem": False, "saveInjectedParameters": False,
                                  "notebookParameters": "{}", "executionTimeoutInMinutes": TASK_TIMEOUT_MIN})})
    log(f"Scheduled '{NOTEBOOK_TITLE}' at {hour}:{minute} UTC ({TASK_LABEL}), timeout {TASK_TIMEOUT_MIN} min: {res}")

## 8. Forest health indices
| Band | Content |
|---|---|
| 1 | clear flag *c* (1 = not cloud or shadow) |
| 2 | weight *w* = clear and every index valid |
| 3 | NDVI · *w*, NDVI = (B8 − B4)/(B8 + B4): canopy greenness |
| 4 | NDMI · *w*, NDMI = (B8A − B11)/(B8A + B11): canopy water content |
| 5 | NBR · *w*, NBR = (B8A − B12)/(B8A + B12): clearing, burn, harvest |

In [ ]:
STATUS_COLORS = [("Disturbance alert", "#b71c1c"), ("Moisture stress", "#ef6c00"), ("Watch", "#fbc02d"),
                 ("Healthy", "#2e7d32"), ("Low canopy", "#a1887f"), ("No recent data", "#9e9e9e"),
                 ("Pending", "#cfd8dc")]
STATUS_RANK = {"Healthy": 1, "Low canopy": 2, "Watch": 2, "Moisture stress": 3, "Disturbance alert": 4}
GROUP_COLORS = [("Natural forest", "#1b5e20"), ("Dry forest & savannah", "#c0ca33")]
QUALITY = ["Valid", "Cloudy"]

SCHEMA = {
    "layers": [{
        "name": "ForestUnits", "geometry": "esriGeometryPolygon", "display": "forest_name",
        "indexes": ["unit_id", "forest_name", "status", "forest_group"],
        "renderer": unique_renderer("status", STATUS_COLORS, alpha=200),
        "fields": [F("unit_id", "str", 60, "Forest ID"), F("unit_uid", "int", alias="Zone value"),
                   F("forest_name", "str", 120, "Forest"),
                   F("designation", "str", 40, "Type"), F("managed_by", "str", 20, "Managed by"),
                   F("forest_group", "str", 30, domain=[g for g, _ in GROUP_COLORS]),
                   F("area_ha", "dbl", alias="Forest area (ha)"), F("pa_area_ha", "dbl", alias="Whole protected area (ha)"),
                   F("lc_year", "sint", alias="Forest cover year"),
                   F("n_villages", "int", alias="Villages covered"), F("admin_note", "str", 600, "Admin areas"),
                   *ADMIN_FIELDS, *BBOX_FIELDS,
                   F("status", "str", 24, domain=[s for s, _ in STATUS_COLORS]), F("status_rank", "sint"),
                   F("condition", "str", 8, domain=["Good", "Fair", "Poor"]),
                   F("status_note", "str", 1200, "What this means"),
                   F("ndvi", "dbl", alias="NDVI"), F("ndmi", "dbl", alias="NDMI"), F("nbr", "dbl", alias="NBR"),
                   F("clear_pct", "dbl", alias="Clear sky (%)"),
                   F("ndvi_base", "dbl", alias="NDVI 14-month mean"), F("ndvi_anom", "dbl", alias="NDVI vs mean"),
                   F("ndmi_anom", "dbl", alias="NDMI vs mean"), F("ndvi_drop", "dbl", alias="NDVI drop vs last 3 months"),
                   F("nbr_drop", "dbl", alias="NBR drop vs last 3 months"),
                   F("last_obs_date", "date", alias="Last clear image"),
                   F("days_since_obs", "int", alias="Days since clear image"), F("last_check_date", "date", alias="Last image checked"),
                   F("last_check_result", "str", 40), F("obs_14m", "int", alias="Clear images (14 months)"),
                   F("prev_obs_date", "date", alias="Previous clear image"), F("prev_ndvi", "dbl"),
                   F("hist_json", "str", 4000, "Monthly history"), F("updated_utc", "date"), F("last_run_id", "str", 16)]}],
    "tables": [
        {"name": "Observations", "indexes": ["obs_id", "unit_id", "forest_name", "obs_date", "quality"],
         "fields": [F("obs_id", "str", 90), F("unit_id", "str", 60), F("forest_name", "str", 120, "Forest"),
                    F("forest_group", "str", 30), F("province", "str", 50), F("district", "str", 50),
                    F("area_ha", "dbl"), F("obs_date", "date", alias="Image date"), F("acq_day", "str", 10),
                    F("tiles", "str", 300), F("n_tiles", "sint"), F("tile_cloud_pct", "dbl"),
                    F("quality", "str", 16, domain=QUALITY), F("clear_pct", "dbl"), F("coverage_pct", "dbl"),
                    F("ndvi", "dbl", alias="NDVI"), F("ndmi", "dbl", alias="NDMI"), F("nbr", "dbl", alias="NBR"),
                    F("run_id", "str", 16), F("created_utc", "date"), MEANING]},
        {"name": "AdminLinks", "indexes": ["unit_id", "forest_name", "district", "sector_label", "village_label"],
         "fields": [F("unit_id", "str", 60), F("forest_name", "str", 120, "Forest"), F("forest_group", "str", 30),
                    F("designation", "str", 40), *ADMIN_FIELDS, F("status", "str", 24), F("status_rank", "sint"),
                    F("condition", "str", 8), F("ndvi", "dbl"), F("ndvi_anom", "dbl"), F("ndmi_anom", "dbl"),
                    F("nbr_drop", "dbl"), F("last_obs_date", "date"), F("days_since_obs", "int"), MEANING]},
    ],
}

NDVI_PALETTE = ["#8e2526", "#d84315", "#ef9a3d", "#ffd54f", "#9ccc65", "#43a047", "#1b5e20"]
NDVI_BREAKS = [-1.0, 0.2, 0.35, 0.5, 0.6, 0.7, 0.8, 1.0]
NDVI_LABELS = ["under 0.20: bare soil, rock, cleared or burnt ground",
               "0.20-0.35: very sparse cover, heavily degraded or dry season grass",
               "0.35-0.50: sparse or stressed canopy, thin woodland",
               "0.50-0.60: moderate canopy, recovering or open forest",
               "0.60-0.70: good canopy for woodland, fair for montane forest",
               "0.70-0.80: dense, healthy canopy",
               "over 0.80: very dense closed canopy (typical of Nyungwe and Volcanoes)"]
NBR_BREAKS = [-1.0, 0.0, 0.1, 0.2, 0.3, 0.45, 0.6, 1.0]
NBR_LABELS = ["under 0: burnt, bare or freshly cleared", "0-0.10: very sparse or recently cleared",
              "0.10-0.20: sparse cover", "0.20-0.30: open or stressed canopy", "0.30-0.45: moderate canopy",
              "0.45-0.60: dense canopy", "over 0.60: very dense, moist canopy"]

INDEX_GUIDE = [
    {"code": "OVERLAY_NDVI", "name": "Map overlay: greenness picture (NDVI)",
     "formula": "class of NDVI = (B8 - B4)/(B8 + B4) per 20 m pixel, on clear pixels, cut to each forest",
     "bands": "B4, B8, SCL",
     "meaning": "What the satellite sees inside each forest on its latest clear image: dense canopy, thin canopy or bare "
                "ground, so a clearing, a burn or a degraded edge shows up where it is. The picture is stored in GeoHub "
                "as one imagery layer and replaced when a forest gets a newer clear image.",
     "how_to_read": "Red = bare or cleared, orange and yellow = sparse or stressed, light green = moderate, dark green = "
                    "dense closed canopy. Classes: " + "; ".join(NDVI_LABELS) + ". Clouds, shadows and everything "
                    "outside the forests are transparent.",
     "thresholds": "; ".join(NDVI_LABELS),
     "caveats": "Each forest is shown on the image of its latest valid reading (its 'Last clear image' date), so "
                "neighbouring forests can be from different dates. NDVI saturates above about 0.85 and drops in the dry "
                "season in woodland and savannah without any loss of forest.",
     "reference": "Rouse J.W. et al. (1974)"},
    {"code": "OVERLAY_NBR", "name": "Map overlay: disturbance picture (NBR)",
     "formula": "class of NBR = (B8A - B12)/(B8A + B12) per 20 m pixel, on clear pixels, cut to each forest",
     "bands": "B8A, B12, SCL",
     "meaning": "Where canopy has been removed, burnt or harvested inside each forest, on its latest clear image.",
     "how_to_read": "Red = burnt, bare or freshly cleared; green = dense, moist canopy. Classes: " + "; ".join(NBR_LABELS) + ".",
     "thresholds": "; ".join(NBR_LABELS),
     "caveats": "Dry-season savannah also reads low without any disturbance.",
     "reference": "Key C.H., Benson N.C. (2006) FIREMON"},
    {"code": "NDVI", "name": "Normalized Difference Vegetation Index", "formula": "(B8 - B4) / (B8 + B4)",
     "bands": "B4, B8", "meaning": "Canopy greenness and density, averaged over the clear part of the forest.",
     "how_to_read": "Natural (montane) forest: Good >= 0.70, Fair 0.55-0.70. Dry forest & savannah: Good >= 0.40, "
                    "Fair 0.22-0.40 (strongly seasonal). Below Fair = Poor (shown as 'Low canopy').",
     "thresholds": "per forest group, see how_to_read (provisional)", "caveats": "Saturates in very dense canopy.",
     "reference": "Rouse et al. (1974)"},
    {"code": "NDMI", "name": "Normalized Difference Moisture Index", "formula": "(B8A - B11) / (B8A + B11)",
     "bands": "B8A, B11", "meaning": "Water content of leaves; drops under drought stress before greenness does.",
     "how_to_read": "0.10 or more below the forest's 14-month mean raises 'Moisture stress'.",
     "thresholds": "anomaly <= -0.10", "caveats": "Seasonal in savannah: values are naturally lower in the dry season.",
     "reference": "Gao B.-C. (1996) Remote Sensing of Environment 58:257-266"},
    {"code": "NBR", "name": "Normalized Burn Ratio", "formula": "(B8A - B12) / (B8A + B12)",
     "bands": "B8A, B12", "meaning": "Sensitive to canopy removal, burning and harvesting.",
     "how_to_read": "A drop of 0.10 or more against the previous three months together with an NDVI drop of 0.08 "
                    "raises a 'Disturbance alert'.",
     "thresholds": "NBR drop >= 0.10 and NDVI drop >= 0.08",
     "caveats": "A forest's value is its average: a clearing of a few hectares inside Nyungwe hardly moves it; "
                "local changes show on the disturbance picture.",
     "reference": "Key C.H., Benson N.C. (2006) FIREMON"},
    {"code": "STATUS", "name": "Status", "formula": "Disturbance > Moisture stress > Watch > Healthy / Low canopy",
     "bands": "", "meaning": "Single label for the map.",
     "how_to_read": "Watch = NDVI 0.08 or more below the forest's 14-month mean. 'No recent data' = no clear image for "
                    "45 days. A reading needs 80% of the forest clear (50% for forests over 5,000 ha).",
     "thresholds": "", "caveats": "Screening from satellite imagery, not a field assessment.",
     "reference": ""},
]


def forest_jobs():
    """The four Zonal Statistics jobs run for one image date, over the zone raster of the forests."""
    clear = scl_only(CLEAR_SCL)
    return [("ndvi", masked(nd("B8", "B4"), clear), "MEAN"),
            ("ndmi", masked(nd("B8A", "B11"), clear), "MEAN"),
            ("nbr", masked(nd("B8A", "B12"), clear), "MEAN"),
            ("clear", clear_flag(), "MEAN")]


def stats_geometry(geom):
    """The outline used to cut the overlay picture."""
    rings = geom.get("rings") or []
    if len(rings) <= 60:
        return geom
    order = sorted(rings, key=lambda r: -abs(_signed_area(r)))
    total = sum(abs(_signed_area(r)) for r in order) or 1.0
    kept, acc = [], 0.0
    for ring in order[:STATS_MAX_RINGS]:
        kept.append(ring)
        acc += abs(_signed_area(ring))
        if acc >= STATS_RING_SHARE * total:
            break
    return {"rings": kept}


def min_clear(u):
    """Share of a forest that must be clear for a reading: large forests are rarely clear all at once."""
    return LARGE_MIN_CLEAR if (u.get("area_ha") or 0) >= LARGE_FOREST_HA else 1 - UNIT_MAX_CLOUD


def zonal_values(results, u):
    """The four zonal results for one forest -> an observation, or None when the forest is not in the image."""
    c = results.get("clear", {}).get(u.get("unit_uid"))
    if not c or not c.get("count"):
        return None
    expected = max(1.0, (u.get("area_ha") or 0) * 10000 / ZONE_CELL_M ** 2)
    coverage = min(1.0, c["count"] / expected)
    if coverage < MIN_COVERAGE:
        return None
    vals = {k: rnd((results.get(k, {}).get(u["unit_uid"]) or {}).get("mean"), 4) for k in ("ndvi", "ndmi", "nbr")}
    vals.update(clear_pct=rnd(100 * (c.get("mean") or 0), 1), coverage_pct=rnd(100 * coverage, 1))
    vals["quality"] = "Valid" if (c.get("mean") or 0) >= min_clear(u) and vals["ndvi"] is not None else "Cloudy"
    return vals


def evaluate(u, h):
    last = from_ms(u.get("last_obs_date"))
    group = u.get("forest_group") or "Natural forest"
    if last is None:
        u.update(status="No recent data" if u.get("last_check_date") else "Pending", condition=None)
        u["status_rank"] = 0
        u["status_note"] = unit_note(u)
        return
    day = last.date()
    month = month_key(day)
    u["days_since_obs"] = (TODAY - day).days
    base = hist_baseline(h, "ndvi", month)
    mbase = hist_baseline(h, "ndmi", month)
    ref_ndvi, ref_nbr = hist_recent(h, "ndvi", day), hist_recent(h, "nbr", day)
    ndvi, ndmi, nbr = u.get("ndvi"), u.get("ndmi"), u.get("nbr")
    u["ndvi_base"] = rnd(base)
    u["ndvi_anom"] = rnd(ndvi - base) if None not in (ndvi, base) else None
    u["ndmi_anom"] = rnd(ndmi - mbase) if None not in (ndmi, mbase) else None
    u["ndvi_drop"] = rnd(ref_ndvi - ndvi) if None not in (ndvi, ref_ndvi) else None
    u["nbr_drop"] = rnd(ref_nbr - nbr) if None not in (nbr, ref_nbr) else None
    good, fair = CONDITION_NDVI.get(group, (0.7, 0.55))
    u["condition"] = None if ndvi is None else "Good" if ndvi >= good else "Fair" if ndvi >= fair else "Poor"
    u["obs_14m"] = hist_count(h)
    if u["days_since_obs"] > STALE_DAYS:
        u["status"] = "No recent data"
    elif (u["nbr_drop"] or 0) >= DISTURBANCE_NBR_DROP and (u["ndvi_drop"] or 0) >= DISTURBANCE_NDVI_DROP:
        u["status"] = "Disturbance alert"
    elif u["ndmi_anom"] is not None and u["ndmi_anom"] <= MOISTURE_ANOMALY:
        u["status"] = "Moisture stress"
    elif u["ndvi_anom"] is not None and u["ndvi_anom"] <= WATCH_NDVI_ANOMALY:
        u["status"] = "Watch"
    elif u["condition"] == "Poor":
        u["status"] = "Low canopy"
    else:
        u["status"] = "Healthy"
    u["status_rank"] = STATUS_RANK.get(u["status"], 0)
    u["status_note"] = unit_note(u)


def unit_note(u):
    head = (f"{u.get('forest_name')} ({(u.get('forest_group') or '').lower()}, {u.get('area_ha') or 0:,.0f} ha of forest"
            + (f", {u['district']} District" if u.get("district") else "") + ")")
    if not u.get("last_obs_date"):
        if u.get("last_check_date"):
            return (f"{head}: no clear image yet among the Sentinel-2 dates processed from the last {HISTORY_MONTHS} "
                    f"months; the latest one ({fmt_date(u['last_check_date'])}) was cloudy over this forest. "
                    "Older dates are still being processed.")
        return f"{head}: not yet reached by the {HISTORY_MONTHS}-month image back-fill."
    parts = [f"{head}: last clear image {fmt_date(u['last_obs_date'])} ({ago(u.get('days_since_obs'))})"
             + (f", measured on the {u['clear_pct']:.0f}% of the forest that was clear." if (u.get("clear_pct") or 100) < 90
                else ".")]
    if u["status"] == "No recent data":
        parts.append("Recent images were cloudy, so these values may be out of date.")
    if u.get("ndvi") is not None:
        anom = ""
        if u.get("ndvi_anom") is not None and abs(u["ndvi_anom"]) >= 0.03:
            anom = f", {abs(u['ndvi_anom']):.2f} {'below' if u['ndvi_anom'] < 0 else 'above'} its 14-month average"
        parts.append(f"NDVI {u['ndvi']:.2f} ({(u.get('condition') or '').lower()} condition{anom}).")
    if u["status"] == "Disturbance alert":
        parts.append(f"NBR fell by {u['nbr_drop']:.2f} and NDVI by {u['ndvi_drop']:.2f} compared with the previous "
                     "three months: possible clearing, harvesting or fire. The disturbance picture on the map shows "
                     "where inside the forest the change is.")
    elif u["status"] == "Moisture stress":
        parts.append(f"Canopy moisture (NDMI) is {abs(u['ndmi_anom']):.2f} below normal: drought stress is likely.")
    elif u["status"] == "Watch":
        parts.append("Greenness is clearly below normal for this forest.")
    elif u["status"] == "Low canopy":
        parts.append("Canopy is sparse for this forest type (degraded or recently disturbed)."
                     if u.get("forest_group") == "Natural forest" else
                     "Vegetation is sparse; woodland and savannah greenness drops in the dry seasons, so a low value can "
                     "be seasonal.")
    return " ".join(parts)[:1200]


def obs_meaning(a):
    """Plain-language reading of one observation row."""
    head = f"{a.get('forest_name') or a.get('unit_id')}, {a.get('acq_day')}: "
    if a.get("quality") != "Valid":
        return head + (f"only {a.get('clear_pct') or 0:.0f}% of the forest was clear of cloud and shadow on this date, "
                       "so no values were taken.")
    group = a.get("forest_group") or "Natural forest"
    good, fair = CONDITION_NDVI.get(group, (0.7, 0.55))
    ndvi = a.get("ndvi")
    if ndvi is None:
        return head + "no value."
    cond = ("good canopy (dense, healthy)" if ndvi >= good else "fair canopy (thinner or stressed)" if ndvi >= fair
            else "poor canopy (sparse, cleared or dry)")
    parts = [f"NDVI {ndvi:.2f} = {cond} (Good >= {good:.2f}, Fair >= {fair:.2f} for {group.lower()})"]
    if a.get("ndmi") is not None:
        parts.append(f"moisture NDMI {a['ndmi']:.2f} ({'moist' if a['ndmi'] >= 0.2 else 'moderately moist' if a['ndmi'] >= 0 else 'dry'} canopy)")
    if a.get("nbr") is not None:
        parts.append(f"NBR {a['nbr']:.2f} (a drop of 0.10 or more from recent months signals clearing or fire)")
    return head + "; ".join(parts) + f". {a.get('clear_pct') or 0:.0f}% of the forest was clear."


def link_meaning(u):
    return f"{u.get('forest_name')} covers part of this village. " + (u.get("status_note") or "")


OVERLAY_KIND = "forest_overlay"
REGISTRY_KINDS[OVERLAY_KIND] = ("each named forest on its latest clear image (band 1 NDVI, band 2 NBR, clear pixels "
                                "only, cut to the forest), shown on the web map")


def overlay_layers(parts):
    """Greenness (NDVI) and disturbance (NBR) pictures over the forests."""
    saved = registry_get(parts, OVERLAY_KIND) if parts else []
    if saved:
        s = saved[0]
        url, iid, mosaic, when = s["url"], s["item_id"], None, f"each forest's latest clear image, {s.get('label')}"
        ndvi = class_colormap(rf("ExtractBand", {"BandIDs": [0]}), NDVI_BREAKS, NDVI_PALETTE)
        nbr = class_colormap(rf("ExtractBand", {"BandIDs": [1]}), NBR_BREAKS, NDVI_PALETTE)
    else:
        url, iid, mosaic, when = S2["url"], S2_ITEM_ID, s2_live_mosaic(by="clear"), "clearest recent image, drawn live"
        ndvi = class_colormap(nd_raw("B8", "B4"), NDVI_BREAKS, NDVI_PALETTE, scl_only(CLEAR_SCL))
        nbr = class_colormap(nd_raw("B8A", "B12"), NBR_BREAKS, NDVI_PALETTE, scl_only(CLEAR_SCL))
    zoom = " - zoom to a forest to see it"
    return [op_imagery("ndvi_overlay", f"Greenness overlay, NDVI ({when}): red = bare, green = dense{zoom}", url, iid,
                       ndvi, mosaic, visible=True, opacity=0.95, min_scale=OVERLAY_MIN_SCALE),
            op_imagery("nbr_overlay", f"Disturbance overlay, NBR ({when}): red = burnt or cleared{zoom}", url, iid,
                       nbr, mosaic, visible=False, opacity=0.95, min_scale=OVERLAY_MIN_SCALE)]


def overlay_stack():
    """Band 1 = NDVI, band 2 = NBR; both only on clear pixels."""
    clear = scl_only(CLEAR_SCL)
    return composite([masked(nd_raw("B8", "B4"), clear), masked(nd_raw("B8A", "B12"), clear)])


def export_image(params, attempts=3):
    """One exportImage request to the Sentinel-2 service -> GeoTIFF bytes (an error answer is raised)."""
    import requests
    token = getattr(gis._con, "token", None) or s2_token()
    last = None
    for attempt in range(attempts):
        try:
            res = requests.post(S2["url"] + "/exportImage", data=dict(params, f="image", token=token), timeout=900)
            if res.content[:4] in (b"II*\x00", b"MM\x00*"):
                return res.content
            last = res.text[:300]
        except Exception as exc:
            last = str(exc)[:300]
        time.sleep(10 * (attempt + 1))
    raise RuntimeError(f"exportImage failed: {last}")


def export_unit(unit_id, geom_utm, oids, folder):
    """GeoTIFF pieces of one forest on its latest clear date: 20 m cells on a grid shared by all forests (UTM 35S),
    cut to the forest, at most OVERLAY_CHUNK_PX pixels a side."""
    cell = OVERLAY_CELL_M
    b = bbox(geom_utm)
    x0, y0 = int(math.floor(b[0] / cell) * cell), int(math.floor(b[1] / cell) * cell)
    x1, y1 = int(math.ceil(b[2] / cell) * cell), int(math.ceil(b[3] / cell) * cell)
    step = OVERLAY_CHUNK_PX * cell
    rule = rf("Clip", {"ClippingGeometry": dict(stats_geometry(geom_utm), spatialReference={"wkid": WORK_WKID}),
                       "ClipType": 1, "Raster": overlay_stack()})
    safe = re.sub(r"\W", "_", str(unit_id))
    paths = []
    for i, cx in enumerate(range(x0, x1, step)):
        for j, cy in enumerate(range(y0, y1, step)):
            ex, ey = min(cx + step, x1), min(cy + step, y1)
            data = export_image({
                "bbox": f"{cx},{cy},{ex},{ey}", "bboxSR": WORK_WKID, "imageSR": WORK_WKID,
                "size": f"{(ex - cx) // cell},{(ey - cy) // cell}", "format": "tiff", "pixelType": "F32",
                "noData": OVERLAY_NODATA, "noDataInterpretation": "esriNoDataMatchAny",
                "interpolation": "RSP_NearestNeighbor", "compression": "LZ77",
                "renderingRule": json.dumps(rule), "mosaicRule": json.dumps(lock_rule(oids))})
            path = os.path.join(folder, f"{safe}_{i}_{j}.tif")
            with open(path, "wb") as fh:
                fh.write(data)
            paths.append(path)
    return paths


def overlay_signature(units):
    """Which image each forest shows. A new picture is made only when this changes."""
    import hashlib
    pairs = sorted((u["unit_id"], u["last_obs_date"]) for u in units)
    return hashlib.sha1(json.dumps(pairs).encode()).hexdigest()[:12]


def save_overlay(parts):
    """Save a new overlay picture when a forest has a newer clear image than the saved one shows."""
    if not MAKE_OVERLAY or not parts.get("RasterRegistry") or not parts.get("ForestUnits"):
        return ""
    feats = query_all(parts["ForestUnits"], fields="unit_id,forest_name,last_obs_date", geometry=True, out_sr=WORK_WKID,
                      max_offset=5)
    units = [dict(lower_keys(f["attributes"]), geometry=f.get("geometry")) for f in feats]
    units = [u for u in units if u.get("last_obs_date") and (u.get("geometry") or {}).get("rings")]
    if not units:
        return ""
    saved = registry_get(parts, OVERLAY_KIND)
    if saved and not OVERLAY_FORCE and f"sig {overlay_signature(units)}" in (saved[0].get("note") or ""):
        return "overlay picture unchanged (no forest has a newer clear image)"
    if DRY_RUN:
        log(f"  [dry_run] would save a new overlay picture of {len(units)} forests")
        return ""
    left = TIME_BUDGET_MIN / 0.75 - (dt.datetime.now(UTC) - RUN_START).total_seconds() / 60
    if left < OVERLAY_RESERVE_MIN:
        return "overlay picture postponed to the next run (not enough time left)"
    heartbeat("saving the overlay picture")
    by_day = {}
    for u in units:
        by_day.setdefault(from_ms(u["last_obs_date"]).date().isoformat(), []).append(u)
    oids_of = {}
    for day in by_day:
        start = dt.datetime.fromisoformat(day).replace(tzinfo=UTC)
        oids_of[day] = [s["oid"] for s in find_scenes(start, start + dt.timedelta(days=1))]
    home = "/arcgis/home" if os.path.isdir("/arcgis/home") else os.getcwd()
    folder = os.path.join(home, "forest_overlay", RUN_ID)
    os.makedirs(folder, exist_ok=True)
    log(f"Overlay picture: exporting {len(units)} forests on their latest clear images ({len(by_day)} dates) ...")
    paths, done, failed = [], [], []
    try:
        with ThreadPoolExecutor(OVERLAY_WORKERS) as pool:
            futures = {pool.submit(export_unit, u["unit_id"], u["geometry"], oids_of[day], folder): u
                       for day, group in by_day.items() for u in group if oids_of[day]}
            for fut in as_completed(futures):
                u = futures[fut]
                try:
                    paths += fut.result()
                    done.append(u)
                except Exception as exc:
                    failed.append(f"{u.get('forest_name') or u['unit_id']}: {str(exc)[:150]}")
        if failed:
            log(f"  {len(failed)} forests could not be exported (tried again next run), first: {failed[0]}")
        if not done:
            raise RuntimeError("no forest could be exported")
        days = sorted({from_ms(u["last_obs_date"]).date().isoformat() for u in done})
        name = f"RW_Forest_Overlay_{TODAY:%Y%m%d}_{RUN_ID[:6]}"
        log(f"  publishing {len(paths)} pieces of {len(done)} forests as {name} ...")
        item = ra_publish_files(paths, name, OVERLAY_CELL_M, timeout_min=int(max(15, left - 10)))
    finally:
        import shutil
        shutil.rmtree(folder, ignore_errors=True)
    if not raster_has_pixels(item):
        item.delete()
        raise RuntimeError("the new overlay picture came out empty, so the previous one is kept")
    item.update(item_properties={
        "snippet": (f"Each monitored forest on its latest clear Sentinel-2 image ({days[0]} to {days[-1]}). "
                    "Band 1: NDVI (greenness); band 2: NBR (clearing, fire). Clear pixels only, cut to the forests."),
        "description": (f"Made by the notebook '{NOTEBOOK_TITLE}' and replaced (the old one deleted) whenever a forest "
                        "gets a newer clear image. Drawn in the web map with the NDVI and NBR classes of the IndexGuide "
                        "table." + (f" Missing this time: {'; '.join(failed)[:900]}" if failed else "")),
        "tags": TAGS})
    share_items([item])
    registry_add(parts, OVERLAY_KIND, item, label=f"{days[0]} to {days[-1]}", day=days[-1], cell_m=OVERLAY_CELL_M,
                 note=f"sig {overlay_signature(done)}; {len(done)} forests on {len(days)} dates"
                      + (f"; {len(failed)} missing" if failed else ""), exclusive=True)
    return f"new overlay picture {item.title} ({len(done)} forests on {len(days)} dates)"


def prune_overlays(parts):
    """Keep one overlay picture: once the web map shows the newest, delete the older ones."""
    if DRY_RUN or not parts.get("RasterRegistry"):
        return
    rows = registry_get(parts, OVERLAY_KIND, active=False)
    if len(rows) < 2:
        return
    webmap = find_item(WEBMAP_TITLE, "Web Map")
    shown = json.dumps(webmap.get_data()) if webmap else ""
    if rows[0]["item_id"] not in shown:
        log("  the web map does not show the newest overlay picture yet, so the older one is kept for now")
        return
    for r in rows[1:]:
        old = gis.content.get(r["item_id"]) if r["item_id"] not in shown else None
        if old is None:
            continue
        try:
            old.delete()
            log(f"  deleted the previous overlay picture {old.title}")
        except Exception as exc:
            log(f"  could not delete the previous overlay picture {old.title}: {exc}")

## 9. Named forests and admin links
Forests come from the *Protected Areas* layer (2024 land cover, *Forest* class); each is linked to every village
it covers.

In [ ]:
def display_name(area, kind):
    """'Umuvumba NF' + 'Natural Forest' -> 'Umuvumba Natural Forest'; 'Gishwati' + 'Forest Reserve' -> 'Gishwati
    Forest Reserve'; 'Nyungwe National Park' stays as it is."""
    n = " ".join(str(area).replace("_", "-").split())
    n = re.sub(r"\bNF\b", "Natural Forest", n)
    n = re.sub(r"\bnatural forest\b", "Natural Forest", n, flags=re.I)
    if not re.search(r"\b(Park|Forest|Reserve)\b", n):
        n = f"{n} {' '.join(str(kind).split())}"
    return n


def unit_slug(name):
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")[:60]


def forest_group_of(area):
    return "Dry forest & savannah" if str(area).strip() in DRY_FORESTS else "Natural forest"


def source_units(out_sr):
    """The named forests from the source layer: {unit_id: (attributes, geometry)} in out_sr."""
    offset = 0.0005 if out_sr == 4326 else 50
    out = {}
    for f in query_all(SOURCE_URL, SOURCE_WHERE, "*", geometry=True, out_sr=out_sr, max_offset=offset):
        a = lower_keys(f["attributes"])
        name = display_name(a["areaname"], a["type"])
        out[unit_slug(name)] = (dict(a, forest_name=name), f["geometry"])
    return out


def setup_units(parts):
    url = parts["ForestUnits"]
    if url and count_rows(url) > 0 and not RESYNC_UNITS:
        if not DRY_RUN and not registry_get(parts, "zone_raster"):
            make_zone_raster(parts)
        return
    log("Loading the named forests from the Protected Areas layer ...")
    src = source_units(4326)
    simple = {}
    for f in query_all(SOURCE_URL, SOURCE_WHERE, "areaname,type", geometry=True, out_sr=4326, max_offset=0.0005):
        a = lower_keys(f["attributes"])
        simple[unit_slug(display_name(a["areaname"], a["type"]))] = f["geometry"]
    fids = sorted({a["fid_protected_area"] for a, _ in src.values() if a.get("fid_protected_area") is not None})
    pa_area = {}
    for r in query_all(SOURCE_URL, f"fid_protected_area IN ({sql_list(fids)})", "fid_protected_area,area_ha") if fids else []:
        a = lower_keys(r["attributes"])
        pa_area[a["fid_protected_area"]] = pa_area.get(a["fid_protected_area"], 0) + (a.get("area_ha") or 0)
    items = sorted(src.items(), key=lambda kv: -(kv[1][0].get("area_ha") or 0))
    if MAX_UNITS:
        items = items[:MAX_UNITS]
    units, links = [], []
    for uid, (a, geom) in items:
        near = villages_near(simple.get(uid) or geom)
        mode = lambda k: max({v[k] for v in near}, key=lambda x: sum(1 for v in near if v[k] == x)) if near else None
        district = mode("district")
        prov_row = next((v for v in near if v["district"] == district), {})
        sectors = sorted({v["sector_label"] for v in near})
        single = len({v["village_id"] for v in near}) == 1
        group = forest_group_of(a["areaname"])
        attrs = {"unit_id": uid, "unit_uid": len(units) + 1, "forest_name": a["forest_name"], "designation": a.get("type"),
                 "managed_by": a.get("managing"), "forest_group": group, "area_ha": rnd(a.get("area_ha"), 1),
                 "pa_area_ha": rnd(pa_area.get(a.get("fid_protected_area")), 1),
                 "lc_year": int(a["year"]) if a.get("year") is not None else None,
                 "n_villages": len(near), "province": prov_row.get("province"), "province_id": prov_row.get("province_id"),
                 "district": district, "district_id": prov_row.get("district_id"),
                 "sector": near[0]["sector"] if len(sectors) == 1 else None,
                 "sector_label": sectors[0] if len(sectors) == 1 else f"{len(sectors)} sectors",
                 "village": near[0]["village"] if single else None,
                 "village_label": near[0]["village_label"] if single else f"{len(near)} villages",
                 "admin_note": (f"{len(near)} villages in {len(sectors)} sectors: " + "; ".join(sectors))[:600],
                 "status": "Pending", "status_rank": 0, **bbox_fields(geom)}
        units.append({"geometry": geom, "attributes": attrs})
        links += [{"attributes": {"unit_id": uid, "forest_name": a["forest_name"], "forest_group": group,
                                  "designation": a.get("type"), "status": "Pending", "status_rank": 0, **v}} for v in near]
        log(f"  {a['forest_name']}: {a.get('area_ha') or 0:,.0f} ha of forest, {len(near)} villages, district {district}")
    if url and RESYNC_UNITS and not DRY_RUN:
        for part in ("ForestUnits", "AdminLinks"):
            ids = [lower_keys(r["attributes"])["objectid"] for r in query_all(parts[part], fields="OBJECTID")]
            apply_edits(parts[part], deletes=ids)
    ok, errors = apply_edits(url, adds=units, batch=1)
    links_ok, _ = apply_edits(parts["AdminLinks"], adds=links)
    log(f"Loaded {ok} forests ({len(errors)} failed) and {links_ok} admin links")
    if url and not DRY_RUN and not ok:
        raise RuntimeError(f"no forest could be written: {str(errors[:1])[:300]}")
    if not DRY_RUN:
        make_zone_raster(parts)


def make_zone_raster(parts):
    """The raster every night's statistics are computed over: one 20 m cell per forest number (`unit_uid`)."""
    name = f"RW_ForestV2_Zones_{ZONE_CELL_M}m_{RUN_ID[:6]}"
    log(f"Creating the zone raster {name} (raster analytics) ...")
    item = ra_feature_to_raster(parts["ForestUnits"], "unit_uid", name, ZONE_CELL_M)
    registry_add(parts, "zone_raster", item, label="named forests", cell_m=ZONE_CELL_M,
                 note="Zone raster of ForestUnits.unit_uid", exclusive=True)
    return item

## 10. Nightly processing

In [ ]:
LATEST_KEYS = ["ndvi", "ndmi", "nbr", "clear_pct"]
FINAL_KEYS = ["status", "status_rank", "condition", "status_note", "ndvi", "ndmi", "nbr", "clear_pct", "ndvi_base",
              "ndvi_anom", "ndmi_anom", "ndvi_drop", "nbr_drop",
              "last_obs_date", "days_since_obs", "last_check_date", "last_check_result", "obs_14m", "prev_obs_date",
              "prev_ndvi"]


def apply_obs(u, day_ms, obs):
    """Update a forest's latest/previous values with a valid observation (dates may arrive in any order)."""
    last = u.get("last_obs_date")
    if last is None or day_ms > last:
        if last is not None:
            u.update(prev_obs_date=last, prev_ndvi=u.get("ndvi"))
        u.update({k: obs.get(k) for k in LATEST_KEYS}, last_obs_date=day_ms)
    elif day_ms < last and (u.get("prev_obs_date") is None or day_ms > u["prev_obs_date"]):
        u.update(prev_obs_date=day_ms, prev_ndvi=obs.get("ndvi"))


def load_units(parts):
    url = parts["ForestUnits"]
    if not url:
        src = source_units(4326)
        units = {uid: {"unit_id": uid, "forest_name": a["forest_name"], "forest_group": forest_group_of(a["areaname"]),
                       "designation": a.get("type"), "area_ha": rnd(a.get("area_ha"), 1), **bbox_fields(g)}
                 for uid, (a, g) in src.items()}
        geoms = {uid: g for uid, (a, g) in source_units(3857).items()}
    else:
        units = {lower_keys(f["attributes"])["unit_id"]: lower_keys(f["attributes"]) for f in query_all(url)}
        geoms = {lower_keys(f["attributes"])["unit_id"]: f["geometry"]
                 for f in query_all(url, fields="unit_id", geometry=True, out_sr=3857, max_offset=STATS_SIMPLIFY_M)}
    if MAX_UNITS:
        keep = sorted(units, key=lambda k: -(units[k].get("area_ha") or 0))[:MAX_UNITS]
        units = {k: units[k] for k in keep}
    return units, geoms


def run_monitoring(parts):
    units, geoms = load_units(parts)
    hist = {k: hist_load(u.get("hist_json")) for k, u in units.items()}
    zone_layer = None
    if parts["ForestUnits"]:
        rows = registry_get(parts, "zone_raster")
        if not rows:
            raise RuntimeError("no zone raster is registered; run the notebook again to build it")
        zone_layer = gis.content.get(rows[0]["item_id"]).layers[0]
    prio = coverage_priority(units)
    days, backlog = pending_days(parts["SceneLog"], prio)
    obs_url = parts["Observations"]
    existing = set()
    if days and obs_url:
        for group in chunks(sorted(days), 50):
            existing |= {lower_keys(r["attributes"])["obs_id"]
                         for r in query_all(obs_url, f"acq_day IN ({sql_list(group)})", "obs_id")}
    n_obs = n_valid = n_scenes = 0
    now_ms = to_ms(dt.datetime.now(UTC))
    processed = []
    for i, day in enumerate(day_order(days, prio)):
        if out_of_time(len(days) - i):
            backlog += len(days) - i
            break
        processed.append(day)
        scenes = days[day]
        n_scenes += len(scenes)
        oids = [s["oid"] for s in scenes]
        inside = {uid for uid, u in units.items() if covered(scenes, sample_points(u), need=0.2)}
        if not inside:
            log(f"  {day}: no monitored forests inside the {len(scenes)} tile(s)")
            scenelog_write(parts["SceneLog"], scenes, "no_units")
            continue
        try:
            results = (ra_zonal(zone_layer, forest_jobs(), oids, day_extent(scenes), ZONE_CELL_M, day.replace("-", ""))
                       if zone_layer is not None else dry_run_values(units, geoms, inside, oids))
        except Exception as exc:
            log(f"  {day}: FAILED {exc}")
            scenelog_write(parts["SceneLog"], scenes, "failed", note=str(exc))
            continue
        day_ms = to_ms(dt.date.fromisoformat(day))
        adds, n_cloudy = [], 0
        for uid in inside:
            u = units[uid]
            obs = zonal_values(results, u)
            if obs is None:
                continue
            if u.get("last_check_date") is None or day_ms >= u["last_check_date"]:
                u.update(last_check_date=day_ms, last_check_result=obs["quality"])
            obs_id = f"{uid}_{day}"
            if obs_id in existing:
                continue
            if obs["quality"] != "Valid":
                n_cloudy += 1
            names = {"forest_name": u.get("forest_name"), "forest_group": u.get("forest_group")}
            adds.append({"attributes": {**obs, **names, "obs_id": obs_id, "unit_id": uid,
                                        "meaning": obs_meaning(dict(obs, **names, acq_day=day)),
                                        "province": u.get("province"), "district": u.get("district"),
                                        "area_ha": u.get("area_ha"), "obs_date": day_ms, "acq_day": day,
                                        "tiles": ",".join(sorted({s["tile"] or "" for s in scenes}))[:300],
                                        "n_tiles": len(scenes),
                                        "tile_cloud_pct": rnd(100 * max(s["cloud"] for s in scenes), 1),
                                        "run_id": RUN_ID, "created_utc": now_ms}})
            if obs["quality"] == "Valid":
                n_valid += 1
                apply_obs(u, day_ms, obs)
                hist_add(hist[uid], dt.date.fromisoformat(day), {k: obs[k] for k in ("ndvi", "ndmi", "nbr")})
        if obs_url:
            apply_edits(obs_url, adds=adds, batch=2000)
        n_obs += len(adds)
        finalize_units(parts, units, hist, now_ms, only=inside)
        scenelog_write(parts["SceneLog"], scenes, "processed", len(adds))
        log(f"  {day}: {len(inside)} forests in the tiles, {len(adds)} new rows, "
            f"{sum(1 for a in adds if a['attributes']['quality'] == 'Valid')} valid, {n_cloudy} cloudy")
        heartbeat(f"{i + 1} of {len(days)} image dates; forests with a clear image: "
                  f"{sum(1 for u in units.values() if u.get('last_obs_date'))} of {len(units)}")

    days = {d: days[d] for d in processed}
    alerts = finalize_units(parts, units, hist, now_ms)
    latest = max((s["time"] for ss in days.values() for s in ss), default=None)
    status = "success" if days else "no_new_data"
    summary = (f"{len(days)} new image dates ({n_scenes} tiles), {n_obs} observations written ({n_valid} clear), "
               f"{alerts} disturbance alerts; {sum(1 for u in units.values() if u.get('last_obs_date'))} of "
               f"{len(units)} forests have a clear image" + (f"; {backlog} older dates left for the next run" if backlog else ""))
    return status, summary, {"dates": len(days), "scenes": n_scenes, "units": len(units),
                             "observations": n_obs, "alerts": alerts, "latest_image": latest}


def dry_run_values(units, geoms, inside, oids):
    """Stand-in for the zonal jobs when there is no service yet (a dry run): one image service request per forest."""
    out = {k: {} for k in ("ndvi", "ndmi", "nbr", "clear")}
    for name, rule, _ in forest_jobs():
        res = run_image_stats([(uid, geoms[uid], oids, ZONE_CELL_M) for uid in inside if uid in geoms], rule, WORKERS)
        for uid, st in res.items():
            if isinstance(st, dict) or not st:
                continue
            u = units[uid]
            count = (st[0].get("count") or 0)
            if name == "clear":
                count = min(count, (u.get("area_ha") or 0) * 10000 / ZONE_CELL_M ** 2)
            out[name][u.get("unit_uid")] = {"mean": st[0].get("mean"), "count": count}
    return out


def finalize_units(parts, units, hist, now_ms, only=None):
    """Status and notes for all forests (or only those in `only`), written to ForestUnits and AdminLinks."""
    updates, alerts = [], 0
    for uid, u in units.items():
        if only is not None and uid not in only:
            alerts += u.get("status") == "Disturbance alert"
            continue
        evaluate(u, hist[uid])
        alerts += u["status"] == "Disturbance alert"
        if "objectid" in u:
            updates.append({"attributes": {"OBJECTID": u["objectid"], **{k: u.get(k) for k in FINAL_KEYS},
                                           "hist_json": hist_dump(hist[uid]), "updated_utc": now_ms,
                                           "last_run_id": RUN_ID}})
        elif DRY_RUN and u.get("last_obs_date") and only is None:
            log(f"  [dry_run] {u['status']}: {u['status_note']}")
    if parts["ForestUnits"]:
        apply_edits(parts["ForestUnits"], updates=updates)
    refresh_links(parts, units, only)
    return alerts


def refresh_links(parts, units, only=None):
    url = parts.get("AdminLinks")
    if not url or (only is not None and not only):
        return
    keys = ["status", "status_rank", "condition", "ndvi", "ndvi_anom", "ndmi_anom", "nbr_drop", "last_obs_date",
            "days_since_obs"]
    updates = []
    for r in query_all(url, fields="unit_id"):
        a = lower_keys(r["attributes"])
        u = units.get(a["unit_id"])
        if u and (only is None or a["unit_id"] in only):
            updates.append({"attributes": {"OBJECTID": a["objectid"], **{k: u.get(k) for k in keys},
                                           "meaning": link_meaning(u)[:800]}})
    apply_edits(url, updates=updates)

## 11. Web map and dashboard

In [ ]:
THEME_GREEN = {"dashboardBackgroundColor": "#f1f7f2", "elementOutlineColor": "#d7e7d9", "textColor": "#14301c",
               "tabBorderColor": "#43a047", "activeTabBorderColor": "#2e7d32", "inactiveTabBorderColor": "#c3dbc7"}
HEADER_GREEN = {"backgroundColor": "#1b5e20", "textColor": "#e9f5ea"}
FOREST_LIST_TEXT = (
    '<p><span style="font-size:15px;"><strong>{field/forest_name}</strong></span></p>'
    '<p><span style="font-size:13px;"><strong>Status:</strong> {field/status} &middot; {field/condition} canopy</span></p>'
    '<p><span style="font-size:13px;"><strong>NDVI:</strong> {field/ndvi} ({field/ndvi_anom} vs its 14-month mean)'
    '<br><strong>NBR drop:</strong> {field/nbr_drop} &middot; <strong>Forest:</strong> {field/area_ha} ha</span></p>'
    '<p><span style="font-size:13px;"><strong>Last clear image:</strong> {field/last_obs_date}</span></p>'
    '<p><span style="font-size:12px;">{field/status_note}</span></p>')


def w_list(name, src, text_html, order_by, max_features=10000):
    """The clickable list of forests down the side of the dashboard (as in the water quality dashboard)."""
    w = widget("listWidget", name, src)
    w.update({"datasets": [dset(src, order_by=order_by, max_features=max_features)], "selectionMode": "single",
              "iconType": "none", "text": text_html, "showFilter": True, "showLastUpdate": False})
    return w


def build_apps(item, view, parts):
    if view is None:
        return []
    vid = view.id
    base = view.url.rstrip("/")
    src = {name: (vid, PARTS_ID[name]) for name in PARTS_ID}
    unit_pop = popup("{forest_name}", [("status", "Status", None), ("condition", "Condition", None),
                                       ("ndvi", "NDVI", 2), ("ndvi_anom", "NDVI vs 14-month mean", 2),
                                       ("nbr_drop", "NBR drop (3 months)", 2), ("ndmi_anom", "NDMI vs mean", 2),
                                       ("last_obs_date", "Last clear image", "date"),
                                       ("clear_pct", "Clear sky (%)", 0), ("area_ha", "Forest area (ha)", 0),
                                       ("designation", "Type", None), ("managed_by", "Managed by", None),
                                       ("district", "District", None), ("admin_note", "Admin areas", None)],
                     note_field="status_note")
    layers = s2_live_layers()[:1] + admin_layers() + overlay_layers(parts) + [
        op_feature("units", "Forests (status outline)", f"{base}/{PARTS_ID['ForestUnits']}", vid,
                   unique_renderer("status", STATUS_COLORS, alpha=12, outline_width=2.0), unit_pop, opacity=1.0)]
    tables = [op_table("obs", "Observations", f"{base}/{PARTS_ID['Observations']}", vid),
              op_table("links", "Admin links", f"{base}/{PARTS_ID['AdminLinks']}", vid)]
    webmap = ensure_webmap(layers, tables)
    patch_webmap(webmap, s2_live_layers()[:1] + overlay_layers(parts))

    def make():
        m = w_map(webmap.id)
        U, O, L = src["ForestUnits"], src["Observations"], src["AdminLinks"]
        sels = [w_selector("Province", L, "province"), w_selector("District", L, "district"),
                w_selector("Forest group", U, "forest_group"), w_selector("Status", U, "status")]
        forests = w_list("Forests (click one)", U, FOREST_LIST_TEXT, ["status_rank DESC", "area_ha DESC"])
        valid = {"quality": "Valid"}
        forest_colors = palette(sorted({lower_keys(r["attributes"])["forest_name"]
                                        for r in query_all(parts["ForestUnits"], fields="forest_name")}))
        compare = [w_rank("Greenness against the forest's own 14-month mean (lowest first)", U, "forest_name",
                          "ndvi_anom", stat="min", y_title="NDVI vs mean", digits=3, ascending=True, color="#c62828"),
                   w_rank("Biggest NBR drop against the previous three months (possible clearing or fire)", U,
                          "forest_name", "nbr_drop", stat="max", y_title="NBR drop", digits=3, color="#b71c1c"),
                   w_rank("Greenness now (mean NDVI of the latest clear image)", U, "forest_name", "ndvi",
                          stat="max", y_title="NDVI", digits=3, color="#2e7d32"),
                   w_timeseries("Greenness over time: all forests compared (NDVI)", O, "obs_date", "ndvi",
                                filters=valid, y_title="NDVI", digits=3, split=("forest_name", forest_colors)),
                   w_timeseries("Disturbance index over time: all forests compared (NBR)", O, "obs_date", "nbr",
                                filters=valid, y_title="NBR", digits=3, split=("forest_name", forest_colors)),
                   w_bars("Forests by status", U, "status", colors=STATUS_COLORS)]
        charts = [w_timeseries("Selected forest: greenness (NDVI)", O, "obs_date", "ndvi",
                               filters=valid, y_title="NDVI", digits=3, color="#2e7d32"),
                  w_timeseries("Selected forest: disturbance index (NBR)", O, "obs_date", "nbr",
                               filters=valid, y_title="NBR", digits=3, color="#8d6e63"),
                  w_timeseries("Selected forest: canopy moisture (NDMI)", O, "obs_date", "ndmi",
                               filters=valid, y_title="NDMI", digits=3, color="#1565c0"),
                  w_table("What each image showed (click a forest in the list)", O,
                          [("forest_name", "Forest", None), ("obs_date", "Image date", None), ("quality", "Quality", None),
                           ("clear_pct", "Clear %", 0), ("ndvi", "NDVI", 3), ("nbr", "NBR", 3),
                           ("meaning", "What this means", None)], order_by=["obs_date DESC"]),
                  w_table("Map overlay legend and how to read each index", src["IndexGuide"],
                          [("name", "Index", None), ("how_to_read", "How to read it", None),
                           ("meaning", "What it means", None), ("caveats", "Care", None)],
                          order_by=["sort_order ASC"])]
        inds = [w_indicator("Forests monitored", U),
                w_indicator("Forest area (ha)", U, stat="sum", field="area_ha"),
                w_indicator("Disturbance alerts", U, filters={"status": "Disturbance alert"}),
                w_indicator("Moisture stress", U, filters={"status": "Moisture stress"}),
                w_runlog(src["RunLog"])]
        on_units = [forests] + compare[:3] + [compare[5]]
        on_obs = compare[3:5] + charts[:4]
        cascade_selectors(sels, lambda f: [(w, [(f, f)], None) for w in on_units] +
                          [(m, [(f, f)], "units")] +
                          ([(w, [(f, f)], None) for w in on_obs] if f != "status" else []))
        connect_filter(forests, [(m, [("forest_name", "forest_name")], "units")] +
                       [(c, [("forest_name", "forest_name")], None) for c in charts[:4]], zoom_to=m)
        compare_tabs = ["Greenness vs normal", "Disturbance", "Greenest now", "NDVI compared", "NBR compared", "Status"]
        chart_tabs = ["Selected: NDVI", "Selected: NBR", "Selected: NDMI", "Images explained", "Legend"]
        root = lay_stack("col", [
            lay_stack("row", [lay_stack("col", [lay_item(w, 1 / len(inds)) for w in inds], height=0.21),
                              lay_item(m, height=0.79)], width=0.49),
            lay_stack("row", [lay_tabs([lay_item(c, tab=t) for c, t in zip(compare, compare_tabs)], height=0.5),
                              lay_tabs([lay_item(c, tab=t) for c, t in zip(charts, chart_tabs)], height=0.5)], width=0.32),
            lay_item(forests, width=0.19)])
        return build_dashboard(sels, [m, forests] + compare + charts + inds, root,
                               theme_overrides=THEME_GREEN, header_colors=HEADER_GREEN)

    dash = ensure_dashboard(make)
    return [webmap, dash]

## 12. Run

In [ ]:
def main():
    admin_init()
    s2_init()
    item, parts = ensure_service()
    stage = "setup"
    try:
        if not run_lock(parts, "forest_health"):
            return
        guide_sync(parts)
        setup_units(parts)
        fill_common_meanings(parts)
        fill_meanings(parts["Observations"], obs_meaning)
        stage = "forest_health"
        view = ensure_view(item)
        if ENSURE_APPS and find_item(WEBMAP_TITLE, "Web Map") is None:
            try:
                build_apps(item, view, parts)
            except Exception as exc:
                log(f"WARNING: could not build the map and dashboard yet: {exc}")
        status, summary, counts = run_monitoring(parts)
        try:
            note = save_overlay(parts)
        except Exception as exc:
            note = f"overlay picture failed, the previous one is kept ({str(exc)[:150]})"
            log(f"WARNING: {note}")
        summary += f"; {note}" if note else ""
        apps = []
        if ENSURE_APPS:
            try:
                apps = build_apps(item, view, parts)
            except Exception as exc:
                log(f"WARNING: web map / dashboard step failed: {exc}")
        try:
            prune_overlays(parts)
        except Exception as exc:
            log(f"WARNING: could not remove the previous overlay picture: {exc}")
        share_items([x for x in [view] + apps if x is not None])
        ensure_task()
        runlog_write(parts, stage, status, summary, **counts)
    except Exception as exc:
        runlog_write(parts, stage, "failed", f"{exc}\n{traceback.format_exc()[-2000:]}")
        raise


main()